In [ ]:
# Cell 1 — Konfigürasyon + Veri Üretimi + Analiz Motoru
# ╔══════════════════════════════════════════════════════════════════╗
# ║  PART 1 — KONFİGÜRASYON · VERİ ÜRETİMİ · ANALİZ MOTORU        ║
# ║  2 Dönem: 2025.09 (Sep) · 2026.03 (Mar)                        ║
# ║  Bu hücreyi bir kez çalıştır — tüm hesaplamalar burada hazırlanır║
# ╚══════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════
# § 1 · KÜTÜPHANELER & MASTER CONFIG
# ══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import math as _math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
import seaborn as sns
import networkx as nx
import warnings
from collections import Counter as _Counter

warnings.filterwarnings("ignore")

# ─── 1.1 Analiz Parametreleri ────────────────────────────────────
X_DAYS      = 7
N_MUSTERI   = 1000
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ─── 1.2 Dönem Tanımları  ▶▶  SADECE 2 DÖNEM ◀◀ ─────────────────
DONEMLER     = ["2025.09", "2026.03"]
DONEM_SIRASI = DONEMLER
DONEM_ARALIK = {
    "2025.09": ("2025-09-01", "2025-09-30"),
    "2026.03": ("2026-03-01", "2026-03-31"),
}
ISLEM_BASLANGIC = pd.Timestamp("2025-08-24")   # 7 gün öncesi pencere
ISLEM_BITIS     = pd.Timestamp("2026-04-07")   # 7 gün sonrası pencere

# ─── 1.3 Segment Konfigürasyonu ──────────────────────────────────
SEGMENT_SIRASI = [
    "Bireysel_Standart", "Bireysel_Premium", "Bireysel_Elite",
    "KOBİ", "KOBİ_Orta", "KOBİ_Büyük",
    "Kurumsal", "Kurumsal_Premium", "Private_Banking", "Ultra_HNW",
]
SEG_ISIMLER = SEGMENT_SIRASI
SEG_ADETLER = [300, 210, 120, 120, 90, 60, 50, 30, 15, 5]

SEG_FON_ORT = {
    "Bireysel_Standart":    15_000,
    "Bireysel_Premium":     75_000,
    "Bireysel_Elite":      300_000,
    "KOBİ":                250_000,
    "KOBİ_Orta":         1_000_000,
    "KOBİ_Büyük":        4_000_000,
    "Kurumsal":          2_000_000,
    "Kurumsal_Premium": 10_000_000,
    "Private_Banking":  25_000_000,
    "Ultra_HNW":        40_000_000,
}
SEG_GUN_ISLEM = {
    "Bireysel_Standart": 0.8,  "Bireysel_Premium": 1.5, "Bireysel_Elite": 2.5,
    "KOBİ": 3.0,               "KOBİ_Orta": 4.5,        "KOBİ_Büyük": 6.0,
    "Kurumsal": 6.0,           "Kurumsal_Premium": 8.0, "Private_Banking": 10.0,
    "Ultra_HNW": 12.0,
}
SINYAL_PROB = 0.35

# ─── 1.4 Ürün & Renk Konfigürasyonu ─────────────────────────────
URUNLER = ["Vadesiz", "Vadeli", "Yatırım", "Döviz", "Kredi"]

_SEG_CLR = ["#2563EB","#7C3AED","#059669","#DC2626","#D97706",
            "#0891B2","#BE185D","#7C2D12","#1D4ED8","#065F46"]
_SEG_CLR_LITE = ["#BFDBFE","#DDD6FE","#A7F3D0","#FECACA","#FDE68A",
                 "#CFFAFE","#FBCFE8","#FED7AA","#DBEAFE","#D1FAE5"]
SEG_RENK      = {s: _SEG_CLR[i]      for i, s in enumerate(SEGMENT_SIRASI)}
SEG_RENK_ACIK = {s: _SEG_CLR_LITE[i] for i, s in enumerate(SEGMENT_SIRASI)}
SEG_RENK_LIST = [SEG_RENK[s] for s in SEGMENT_SIRASI]

URUN_RENKLER = {
    "Vadesiz": "#3B82F6", "Vadeli": "#F59E0B",
    "Yatırım": "#10B981", "Döviz":  "#8B5CF6", "Kredi": "#EF4444",
}
URUN_RENK_LIST = [URUN_RENKLER[u] for u in URUNLER]
YON_RENK = {"Giriş": "#059669", "Çıkış": "#DC2626"}

# ─── 1.5 Bakiye Dilimi ───────────────────────────────────────────
BAKIYE_PERCENTIL        = [0.33, 0.67]
BAKIYE_DILIM_ETIKETLERI = ["Düşük", "Orta", "Yüksek"]
BAKIYE_DILIMLERI        = BAKIYE_DILIM_ETIKETLERI
_BAKIYE_CLR = ["#FCA5A5","#93C5FD","#6EE7B7","#FDE68A","#DDD6FE",
               "#A7F3D0","#FBCFE8","#FED7AA","#BAE6FD","#BBF7D0"]
BAKIYE_RENK = {l: _BAKIYE_CLR[i] for i, l in enumerate(BAKIYE_DILIM_ETIKETLERI)}

# ─── 1.6 Grafik Stil ─────────────────────────────────────────────
FIG_BG = "#FFFFFF"; AXES_BG = "#F8FAFC"; GRID_CLR = "#E2E8F0"
TEXT_CLR = "#1E293B"; SPINE_CLR = "#94A3B8"; TITLE_CLR = "#0F172A"

def style_axes(ax, grid=True, despine=True):
    ax.set_facecolor(AXES_BG)
    if grid:
        ax.grid(True, color=GRID_CLR, linewidth=0.6, alpha=0.9, zorder=0)
        ax.set_axisbelow(True)
    if despine:
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color(SPINE_CLR); ax.spines["bottom"].set_color(SPINE_CLR)
    else:
        for sp in ax.spines.values(): sp.set_color(SPINE_CLR); sp.set_linewidth(0.7)
    ax.tick_params(colors=TEXT_CLR, labelsize=8.5)
    ax.xaxis.label.set_color(TEXT_CLR); ax.yaxis.label.set_color(TEXT_CLR)
    ax.title.set_color(TITLE_CLR)

def style_fig(fig, title=None, subtitle=None):
    fig.patch.set_facecolor(FIG_BG)
    if title:
        fig.suptitle(title, fontsize=14, fontweight="bold", color=TITLE_CLR,
                     x=0.5, y=0.99, va="top")
    if subtitle:
        fig.text(0.5, 0.965, subtitle, ha="center", fontsize=9.5,
                 color=SPINE_CLR, style="italic")

def _short_label(s, maxlen=10):
    for o, n in [("Bireysel_","Bir."),("_Standart","_Std"),("_Premium","_Prm"),
                 ("_Elite","_Elt"),("KOBİ_Orta","KOBI.Ort"),("KOBİ_Büyük","KOBI.Byk"),
                 ("Kurumsal","Kur."),("Private_Banking","Priv.Bnk"),("Ultra_HNW","Ultra")]:
        s = s.replace(o, n)
    return s[:maxlen]

SEG_SHORT = [_short_label(s) for s in SEGMENT_SIRASI]

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":10,"axes.titlesize":11,
    "axes.titleweight":"bold","axes.titlepad":10,"axes.labelsize":9,
    "xtick.labelsize":8.5,"ytick.labelsize":8.5,"legend.fontsize":8.5,
    "legend.framealpha":0.93,"legend.edgecolor":"#94A3B8","legend.borderpad":0.6,
    "figure.dpi":110,"savefig.dpi":150,"figure.facecolor":FIG_BG,
})

SHOW_ANNOTATIONS = True; SHOW_CI = True; SAVE_CHARTS = False; CHART_DIR = "charts"
NET_MIN_EDGE_PCT = 3.0; NET_NODE_SCALE = 5000; NET_EDGE_SCALE = 15
NET_SEQ_MIN_EDGE = 10;  NET_SEQ_MIN_PCT = 2.0
DF_ALIM = "alim_df"; DF_SATIM = "satim_df"; DF_ISLEM = "islem_df"
DF_BAKIYE = "bakiye_df"; DF_MUSTERI = "musteri_df"

print("✅ § 1 Config yüklendi")
print(f"   Dönemler: {DONEM_SIRASI}")
print(f"   Segmentler ({len(SEGMENT_SIRASI)}): {', '.join(SEGMENT_SIRASI)}")


# ══════════════════════════════════════════════════════════════════
# § 2 · VERİ ÜRETİMİ
# ══════════════════════════════════════════════════════════════════
np.random.seed(RANDOM_SEED)

musteri_ids    = [f"MUS{str(i).zfill(5)}" for i in range(1, N_MUSTERI + 1)]
segment_dizisi = np.repeat(SEG_ISIMLER, SEG_ADETLER)
musteri_df     = pd.DataFrame({"musteri_id": musteri_ids, "musteri_segmenti": segment_dizisi})
seg_map        = musteri_df.set_index("musteri_id")["musteri_segmenti"].to_dict()

def lognorm_tutar(ort_dizi, sigma=0.55):
    ort_dizi = np.maximum(ort_dizi, 1.0)
    return np.round(
        np.exp(np.log(ort_dizi) + np.random.randn(len(ort_dizi)) * sigma) / 100) * 100

# ── Alım DF ──────────────────────────────────────────────────────
alim_kayitlar = []
for donem in DONEMLER:
    bas = pd.Timestamp(DONEM_ARALIK[donem][0])
    bit = pd.Timestamp(DONEM_ARALIK[donem][1])
    gun_sayisi = (bit - bas).days + 1
    for seg in SEG_ISIMLER:
        seg_musteriler = musteri_df[musteri_df["musteri_segmenti"] == seg]["musteri_id"].values
        n_katilan = int(len(seg_musteriler) * np.random.uniform(0.55, 0.65))
        katilan   = np.random.choice(seg_musteriler, size=n_katilan, replace=False)
        for m in katilan:
            n_islem  = np.random.randint(1, 5)
            gun_ofset = np.random.randint(0, gun_sayisi, size=n_islem)
            tarihler  = bas + pd.to_timedelta(gun_ofset, unit="D")
            tutarlar  = lognorm_tutar(np.full(n_islem, SEG_FON_ORT[seg]))
            for k in range(n_islem):
                alim_kayitlar.append({"musteri_id": m, "tarih": tarihler[k],
                    "donem": donem, "alim_tutari": tutarlar[k],
                    "islem_adeti": 1, "alim_flg": 1})

alim_df = (pd.DataFrame(alim_kayitlar)
           .assign(tarih=lambda d: pd.to_datetime(d["tarih"]))
           .sort_values(["musteri_id","tarih"]).reset_index(drop=True))

# ── Satım DF ─────────────────────────────────────────────────────
satim_kayitlar = []
for donem in DONEMLER:
    bit       = pd.Timestamp(DONEM_ARALIK[donem][1])
    alim_uniq = (alim_df[alim_df["donem"] == donem]
                 .sort_values("tarih").drop_duplicates("musteri_id", keep="first"))
    satim_yapanlar = alim_uniq.sample(frac=0.40, random_state=RANDOM_SEED)
    for _, row in satim_yapanlar.iterrows():
        m_id = row["musteri_id"]; seg = seg_map[m_id]
        base_t = row["tarih"]; kalan = int((bit - base_t).days)
        if kalan < 1: continue
        max_offset = min(kalan, 45); n_islem = np.random.randint(1, 4)
        tutarlar = lognorm_tutar(
            np.full(n_islem, SEG_FON_ORT[seg] * np.random.uniform(0.60, 0.90)))
        for k in range(n_islem):
            offset = np.random.randint(1, max_offset + 1)
            t = min(base_t + pd.Timedelta(days=int(offset)), bit)
            satim_kayitlar.append({"musteri_id": m_id, "tarih": t, "donem": donem,
                "satim_tutari": tutarlar[k], "islem_adeti": 1, "satim_flg": 1})

satim_df = (pd.DataFrame(satim_kayitlar)
            .assign(tarih=lambda d: pd.to_datetime(d["tarih"]))
            .sort_values(["musteri_id","tarih"]).reset_index(drop=True))

# ── İşlem DF ─────────────────────────────────────────────────────
URUN_ISLEM = pd.DataFrame([
    ("Vadesiz","EFT","Giriş",0.15),("Vadesiz","EFT","Çıkış",0.15),
    ("Vadesiz","Havale","Giriş",0.08),("Vadesiz","Havale","Çıkış",0.08),
    ("Vadesiz","ATM_Çekim","Çıkış",0.06),("Vadesiz","Fatura_Ödeme","Çıkış",0.05),
    ("Vadeli","Vadeli_Açılış","Çıkış",0.04),("Vadeli","Vadeli_Kapanış","Giriş",0.04),
    ("Yatırım","Hisse_Alım","Çıkış",0.05),("Yatırım","Hisse_Satım","Giriş",0.04),
    ("Yatırım","TahvilBono_Alım","Çıkış",0.03),("Yatırım","TahvilBono_Satım","Giriş",0.03),
    ("Yatırım","Repo_Giriş","Giriş",0.03),
    ("Döviz","Döviz_Alım","Çıkış",0.06),("Döviz","Döviz_Satım","Giriş",0.05),
    ("Kredi","Kredi_Ödemesi","Çıkış",0.06),("Kredi","Kredi_Kullanımı","Giriş",0.03),
], columns=["urun_grubu","islem_turu","islem_yonu","agirlik"])

AGIRLIKLAR   = (URUN_ISLEM["agirlik"] / URUN_ISLEM["agirlik"].sum()).values
TOPLAM_GUN   = (ISLEM_BITIS - ISLEM_BASLANGIC).days + 1
TARIH_DIZISI = pd.date_range(ISLEM_BASLANGIC, ISLEM_BITIS, freq="D")

lambda_dizisi = musteri_df["musteri_segmenti"].map(SEG_GUN_ISLEM).values
toplam_islem  = np.random.poisson(lambda_dizisi * TOPLAM_GUN)
musteri_rep   = musteri_df.loc[musteri_df.index.repeat(toplam_islem)].reset_index(drop=True)
N_islem       = len(musteri_rep)

rand_tarih_idx = np.random.randint(0, TOPLAM_GUN, size=N_islem)
rand_tarihler  = TARIH_DIZISI[rand_tarih_idx]
tx_idx         = np.random.choice(len(URUN_ISLEM), size=N_islem, p=AGIRLIKLAR)
seg_ort_arr    = musteri_rep["musteri_segmenti"].map(
    {s: SEG_FON_ORT[s] * 0.10 for s in SEG_FON_ORT}).values
tutarlar_arr   = lognorm_tutar(seg_ort_arr, sigma=0.6)

islem_df = pd.DataFrame({
    "musteri_id":       musteri_rep["musteri_id"].values,
    "musteri_segmenti": musteri_rep["musteri_segmenti"].values,
    "islem_tarihi":     rand_tarihler,
    "islem_yonu":       URUN_ISLEM["islem_yonu"].values[tx_idx],
    "urun_grubu":       URUN_ISLEM["urun_grubu"].values[tx_idx],
    "islem_turu":       URUN_ISLEM["islem_turu"].values[tx_idx],
    "islem_tutari":     tutarlar_arr,
    "islem_adeti":      1,
})

# Davranışsal sinyal
def sinyal_uret(event_df, gun_ofset, yon, urun, tur, tarih_kolon="tarih"):
    tmp = event_df[["musteri_id", tarih_kolon]].copy()
    tmp["islem_tarihi"] = pd.to_datetime(tmp[tarih_kolon]) + pd.Timedelta(days=gun_ofset)
    tmp = tmp[(tmp["islem_tarihi"] >= ISLEM_BASLANGIC) &
              (tmp["islem_tarihi"] <= ISLEM_BITIS)].copy()
    if tmp.empty: return pd.DataFrame()
    mask = np.random.random(len(tmp)) < SINYAL_PROB
    tmp  = tmp[mask].copy()
    if tmp.empty: return pd.DataFrame()
    tmp["musteri_segmenti"] = tmp["musteri_id"].map(seg_map)
    tmp["islem_yonu"] = yon; tmp["urun_grubu"] = urun
    tmp["islem_turu"] = tur; tmp["islem_adeti"] = 1
    seg_ort = tmp["musteri_segmenti"].map(
        {s: SEG_FON_ORT[s] * 0.08 for s in SEG_FON_ORT}).values
    tmp["islem_tutari"] = lognorm_tutar(seg_ort, sigma=0.5)
    return tmp[["musteri_id","musteri_segmenti","islem_tarihi",
                "islem_yonu","urun_grubu","islem_turu","islem_tutari","islem_adeti"]]

sinyal_bloklar = []
for g in range(1, 8):
    b = sinyal_uret(alim_df,  -g, "Giriş", "Vadesiz", "EFT")
    if not b.empty: sinyal_bloklar.append(b)
for g in range(1, 8):
    b = sinyal_uret(satim_df,  g, "Çıkış", "Vadesiz", "EFT")
    if not b.empty: sinyal_bloklar.append(b)
if sinyal_bloklar:
    islem_df = (pd.concat([islem_df] + sinyal_bloklar, ignore_index=True)
                .sort_values(["musteri_id","islem_tarihi"]).reset_index(drop=True))

# ── Bakiye DF ────────────────────────────────────────────────────
net_fon = (alim_df.groupby("musteri_id")["alim_tutari"].sum()
           .sub(satim_df.groupby("musteri_id")["satim_tutari"].sum(), fill_value=0))
bakiye_kayitlar = []
for m_id in musteri_ids:
    seg = seg_map[m_id]
    baz = max(net_fon.get(m_id, 0) * 0.8, SEG_FON_ORT[seg] * 0.5)
    for i, donem in enumerate(DONEMLER):
        carpan     = (1 + 0.03 * i) * np.random.uniform(0.85, 1.20)
        bakiye     = round(baz * carpan / 100) * 100
        bakiye_mtd = round(bakiye * np.random.uniform(0.90, 1.10) / 100) * 100
        bakiye_kayitlar.append({
            "musteri_id": m_id, "tarih": pd.Timestamp(DONEM_ARALIK[donem][1]),
            "donem": donem, "fon_bakiye_tutari": max(bakiye, 0),
            "fon_bakiye_mtd_tutari": max(bakiye_mtd, 0),
        })
bakiye_df = (pd.DataFrame(bakiye_kayitlar)
             .assign(tarih=lambda d: pd.to_datetime(d["tarih"]))
             .sort_values(["musteri_id","tarih"]).reset_index(drop=True))

print(f"\n✅ § 2 Veri üretildi: {len(musteri_df):,} müşteri · {len(islem_df):,} işlem · "
      f"{len(alim_df):,} alım · {len(satim_df):,} satım")


# ══════════════════════════════════════════════════════════════════
# § 3 · VERİ KALİTE KONTROLÜ
# ══════════════════════════════════════════════════════════════════
_ns = globals()
alim_df    = _ns[DF_ALIM];   alim_df["tarih"]         = pd.to_datetime(alim_df["tarih"])
satim_df   = _ns[DF_SATIM];  satim_df["tarih"]        = pd.to_datetime(satim_df["tarih"])
islem_df   = _ns[DF_ISLEM];  islem_df["islem_tarihi"] = pd.to_datetime(islem_df["islem_tarihi"])
bakiye_df  = _ns[DF_BAKIYE]; bakiye_df["tarih"]       = pd.to_datetime(bakiye_df["tarih"])
musteri_df = _ns[DF_MUSTERI]

for _df, _col in [(islem_df,"musteri_segmenti"),(musteri_df,"musteri_segmenti")]:
    if _col in _df.columns:
        _df[_col] = pd.Categorical(_df[_col], categories=SEGMENT_SIRASI, ordered=True)

print(f"\n{'§ 3 Kalite Kontrolü':─<50}")
for isim, df in [("alim_df",alim_df),("satim_df",satim_df),
                  ("islem_df",islem_df),("bakiye_df",bakiye_df)]:
    n_null = df.isnull().sum().sum(); n_dup = df.duplicated().sum()
    print(f"  {'✅' if n_null==0 and n_dup==0 else '⚠️'}  {isim:<12} {len(df):>8,} satır  null={n_null}  dup={n_dup}")


# ══════════════════════════════════════════════════════════════════
# § 4 · ANALİZ ENGINE
# ══════════════════════════════════════════════════════════════════

# §4.1 Olay Tablosu
alim_events  = (alim_df.merge(musteri_df[["musteri_id","musteri_segmenti"]], on="musteri_id")
                .rename(columns={"tarih":"event_tarih","alim_tutari":"event_tutari"})
                .assign(event_type="Alım")
                [["musteri_id","musteri_segmenti","event_tarih","event_tutari","donem","event_type"]])
satim_events = (satim_df.merge(musteri_df[["musteri_id","musteri_segmenti"]], on="musteri_id")
                .rename(columns={"tarih":"event_tarih","satim_tutari":"event_tutari"})
                .assign(event_type="Satım")
                [["musteri_id","musteri_segmenti","event_tarih","event_tutari","donem","event_type"]])
events_df = (pd.concat([alim_events, satim_events]).reset_index(drop=True)
             .assign(event_id=lambda d: d.index))

# §4.2 Bakiye Dilimi
bakiye_donem = (bakiye_df[["musteri_id","donem","fon_bakiye_tutari"]]
                .merge(musteri_df[["musteri_id","musteri_segmenti"]], on="musteri_id")
                .rename(columns={"fon_bakiye_tutari":"ort_bakiye"}))

def dilim_ata(grp):
    thresholds = [grp["ort_bakiye"].quantile(p) for p in BAKIYE_PERCENTIL]
    bins = [-np.inf] + thresholds + [np.inf]
    return pd.cut(grp["ort_bakiye"], bins=bins, labels=BAKIYE_DILIM_ETIKETLERI)

bakiye_donem["bakiye_dilimi"] = (
    bakiye_donem.groupby(["musteri_segmenti","donem"], group_keys=False)
                .apply(dilim_ata).astype(str))
musteri_bakiye_dilim = bakiye_donem[["musteri_id","donem","bakiye_dilimi"]].copy()
DILIM_ESLIKLERI = (
    bakiye_donem.groupby(["musteri_segmenti","donem"], observed=True)["ort_bakiye"]
    .quantile(BAKIYE_PERCENTIL).unstack(level=-1)
    .rename(columns={p: f"p{int(p*100)}" for p in BAKIYE_PERCENTIL}))

# §4.3 Zenginleştirilmiş Olay
events_enriched_df = (
    events_df
    .merge(musteri_bakiye_dilim, on=["musteri_id","donem"], how="left")
    .merge(bakiye_df[["musteri_id","donem","fon_bakiye_tutari"]]
           .rename(columns={"fon_bakiye_tutari":"bakiye"}), on=["musteri_id","donem"], how="left"))
events_enriched_df["bakiye_dilimi"] = events_enriched_df["bakiye_dilimi"].fillna(BAKIYE_DILIM_ETIKETLERI[1])

# §4.4 İşlem Penceresi ±X_DAYS
ev_keys = events_df[["event_id","musteri_id","event_tarih","event_type","donem"]].copy()
islem_ev = islem_df.merge(ev_keys, on="musteri_id")
islem_ev["gun_fark"] = (islem_ev["islem_tarihi"] - islem_ev["event_tarih"]).dt.days
islem_window_df = (islem_ev[islem_ev["gun_fark"].between(-X_DAYS, X_DAYS) &
                             (islem_ev["gun_fark"] != 0)].copy())
islem_window_df["pencere"] = islem_window_df["gun_fark"].apply(lambda g: "Pre" if g < 0 else "Post")

pre_buy   = islem_window_df[(islem_window_df["event_type"]=="Alım")  & (islem_window_df["pencere"]=="Pre")]
post_buy  = islem_window_df[(islem_window_df["event_type"]=="Alım")  & (islem_window_df["pencere"]=="Post")]
pre_sell  = islem_window_df[(islem_window_df["event_type"]=="Satım") & (islem_window_df["pencere"]=="Pre")]
post_sell = islem_window_df[(islem_window_df["event_type"]=="Satım") & (islem_window_df["pencere"]=="Post")]

# §4.5 Genel Metrikler
METRIK_GENEL = (
    events_enriched_df.groupby(["musteri_segmenti","event_type"], observed=True)
    .agg(Olay_Adedi=("event_id","count"), Musteri_Sayisi=("musteri_id","nunique"),
         Toplam_Tutar_M=("event_tutari", lambda x: round(x.sum()/1e6,2)),
         Ort_Tutar_K=("event_tutari", lambda x: round(x.mean()/1e3,1)),
         Medyan_Tutar_K=("event_tutari", lambda x: round(x.median()/1e3,1)))
    .reset_index())

# §4.6 Ürün Dağılımı
def urun_pct(df):
    grp = (df.groupby(["musteri_segmenti","urun_grubu"], observed=True).size().reset_index(name="adet"))
    grp["toplam"] = grp.groupby("musteri_segmenti", observed=True)["adet"].transform("sum")
    grp["pct"]    = grp["adet"] / grp["toplam"] * 100
    return (grp.pivot(index="musteri_segmenti", columns="urun_grubu", values="pct").fillna(0)
              .reindex(SEGMENT_SIRASI).reindex(columns=URUNLER, fill_value=0))

URUN_PRE_BUY   = urun_pct(pre_buy)
URUN_POST_BUY  = urun_pct(post_buy)
URUN_PRE_SELL  = urun_pct(pre_sell)
URUN_POST_SELL = urun_pct(post_sell)

# §4.7 Giriş Oranı & Net Akış
def giris_pct(df):
    return (df.groupby("musteri_segmenti", observed=True)["islem_yonu"]
              .apply(lambda x: (x=="Giriş").mean()*100).round(1).reindex(SEGMENT_SIRASI))
def net_akis(df):
    gin = (df[df["islem_yonu"]=="Giriş"]
           .groupby(["musteri_segmenti","event_id"], observed=True)["islem_tutari"].sum())
    cik = (df[df["islem_yonu"]=="Çıkış"]
           .groupby(["musteri_segmenti","event_id"], observed=True)["islem_tutari"].sum())
    return (gin.sub(cik, fill_value=0).reset_index().rename(columns={"islem_tutari":"net"})
               .groupby("musteri_segmenti", observed=True)["net"].mean().reindex(SEGMENT_SIRASI))

GIRIS_PRE_BUY   = giris_pct(pre_buy);   GIRIS_POST_BUY  = giris_pct(post_buy)
GIRIS_PRE_SELL  = giris_pct(pre_sell);  GIRIS_POST_SELL = giris_pct(post_sell)
NET_PRE_BUY     = net_akis(pre_buy);    NET_POST_BUY    = net_akis(post_buy)
NET_PRE_SELL    = net_akis(pre_sell);   NET_POST_SELL   = net_akis(post_sell)

# §4.8 Bakiye Dilimi × Segment
ev_bd = (events_enriched_df[["event_id","bakiye_dilimi"]].drop_duplicates("event_id")
         .assign(bakiye_dilimi=lambda d: d["bakiye_dilimi"].astype(str)))

def bd_pivot(df, val="islem_tutari"):
    d2 = df.merge(ev_bd, on="event_id", how="left")
    d2["bakiye_dilimi"] = pd.Categorical(
        d2["bakiye_dilimi"].astype(str).fillna(BAKIYE_DILIM_ETIKETLERI[1]),
        categories=BAKIYE_DILIM_ETIKETLERI, ordered=True)
    if val == "count":
        pv = d2.groupby(["musteri_segmenti","bakiye_dilimi"], observed=True).size().unstack(fill_value=0)
    else:
        pv = d2.groupby(["musteri_segmenti","bakiye_dilimi"], observed=True)[val].mean().unstack(fill_value=0)
    return pv.reindex(SEGMENT_SIRASI).reindex(columns=BAKIYE_DILIM_ETIKETLERI, fill_value=0)

BD_ALIM_COUNT  = bd_pivot(pd.concat([pre_buy,  post_buy]),  "count")
BD_ALIM_TUTAR  = bd_pivot(pd.concat([pre_buy,  post_buy]))
BD_SATIM_COUNT = bd_pivot(pd.concat([pre_sell, post_sell]), "count")
BD_SATIM_TUTAR = bd_pivot(pd.concat([pre_sell, post_sell]))

# §4.9 Geçiş Matrisi
def gecis_mat(event_type):
    df_e  = islem_window_df[islem_window_df["event_type"]==event_type]
    pre_u = (df_e[df_e["pencere"]=="Pre"]
             .groupby(["event_id","urun_grubu"], observed=True).size()
             .reset_index(name="n").sort_values("n", ascending=False)
             .drop_duplicates("event_id")[["event_id","urun_grubu"]].rename(columns={"urun_grubu":"pre_urun"}))
    post_u = (df_e[df_e["pencere"]=="Post"]
              .groupby(["event_id","urun_grubu"], observed=True).size()
              .reset_index(name="n").sort_values("n", ascending=False)
              .drop_duplicates("event_id")[["event_id","urun_grubu"]].rename(columns={"urun_grubu":"post_urun"}))
    gecis = pre_u.merge(post_u, on="event_id", how="inner")
    mat   = (gecis.groupby(["pre_urun","post_urun"], observed=True).size()
                  .unstack(fill_value=0)
                  .reindex(index=URUNLER, columns=URUNLER, fill_value=0))
    return (mat.div(mat.sum(axis=1).clip(lower=1), axis=0)*100).round(1)

GECIS_ALIM  = gecis_mat("Alım")
GECIS_SATIM = gecis_mat("Satım")

# §4.10 Sadakat
musteri_donem_alim = alim_df.groupby("musteri_id")["donem"].nunique()
SADAKAT = (
    musteri_donem_alim.reset_index().rename(columns={"donem":"n_donem"})
    .merge(musteri_df, on="musteri_id")
    .groupby("musteri_segmenti", observed=True).agg(
        Ort_Donem_Alim=("n_donem","mean"),
        Cok_Donem_Pct=("n_donem", lambda x: (x > 1).mean()*100),
        Tek_Donem_Pct=("n_donem", lambda x: (x == 1).mean()*100))
    .round(1).reindex(SEGMENT_SIRASI))

# §4.11 Günlük Akış
def gunluk_ortalama(pre_df, post_df):
    pre_g  = pre_df.groupby("gun_fark",  observed=True)["islem_tutari"].mean()
    post_g = post_df.groupby("gun_fark", observed=True)["islem_tutari"].mean()
    return pre_g.combine_first(post_g).reindex(range(-X_DAYS, X_DAYS+1)).fillna(0)

GUNLUK_ALIM  = gunluk_ortalama(pre_buy,  post_buy)
GUNLUK_SATIM = gunluk_ortalama(pre_sell, post_sell)

# §4.12 Behavioral Scoring
freq_pre = (pre_buy.groupby(["musteri_segmenti","event_id"], observed=True).size()
            .reset_index(name="n").groupby("musteri_segmenti", observed=True)["n"].mean()
            .reindex(SEGMENT_SIRASI).fillna(0))
sadakat_pct = (
    musteri_donem_alim.reset_index().rename(columns={"donem":"n_donem"})
    .merge(musteri_df, on="musteri_id")
    .groupby("musteri_segmenti", observed=True)
    .apply(lambda x: (x["n_donem"] > 1).mean()*100).round(1)
    .reindex(SEGMENT_SIRASI).fillna(0))
DAVRANIS_SKOR = pd.DataFrame({
    "Pre_Buy_Giriş_%": GIRIS_PRE_BUY, "Post_Buy_Giriş_%": GIRIS_POST_BUY,
    "Pre_Sell_Giriş_%": GIRIS_PRE_SELL, "Post_Sell_Giriş_%": GIRIS_POST_SELL,
    "Pre_Buy_Frekans": freq_pre, "Sadakat_%": sadakat_pct,
}).round(1)
DAVRANIS_SKOR["Aktivite_Skoru"] = (
    DAVRANIS_SKOR["Pre_Buy_Giriş_%"]*0.25 + DAVRANIS_SKOR["Post_Buy_Giriş_%"]*0.20 +
    DAVRANIS_SKOR["Pre_Buy_Frekans"]*5.0  + DAVRANIS_SKOR["Sadakat_%"]*0.30
).clip(0, 100).round(1)

# §4.13 Penetrasyon
seg_toplam  = musteri_df.groupby("musteri_segmenti", observed=True).size()
alim_katil  = (alim_df.merge(musteri_df, on="musteri_id")
               .groupby("musteri_segmenti", observed=True)["musteri_id"].nunique())
satim_katil = (satim_df.merge(musteri_df, on="musteri_id")
               .groupby("musteri_segmenti", observed=True)["musteri_id"].nunique())
PENETRASYON = pd.DataFrame({
    "Toplam_Musteri": seg_toplam,
    "Alim_Yapan": alim_katil.reindex(SEGMENT_SIRASI),
    "Satim_Yapan": satim_katil.reindex(SEGMENT_SIRASI),
    "Alim_Pct": (alim_katil/seg_toplam*100).reindex(SEGMENT_SIRASI).round(1),
    "Satim_Pct": (satim_katil/seg_toplam*100).reindex(SEGMENT_SIRASI).round(1),
}).reindex(SEGMENT_SIRASI)

# §4.14 AUM
AUM = (alim_df.merge(musteri_df, on="musteri_id")
       .groupby("musteri_segmenti", observed=True)["alim_tutari"].sum()
       .sub(satim_df.merge(musteri_df, on="musteri_id")
            .groupby("musteri_segmenti", observed=True)["satim_tutari"].sum(), fill_value=0)
       .reindex(SEGMENT_SIRASI))

# §4.15 Sekansiyel Ağ
def _compute_pairs(df):
    d = (df[["musteri_id","islem_tarihi","urun_grubu","islem_tutari","musteri_segmenti"]]
         .sort_values(["musteri_id","islem_tarihi"]).reset_index(drop=True))
    d["next_urun"]    = d.groupby("musteri_id", sort=False)["urun_grubu"].shift(-1)
    d["next_musteri"] = d.groupby("musteri_id", sort=False)["musteri_id"].shift(-1)
    return (d.dropna(subset=["next_urun"]).pipe(lambda x: x[x["musteri_id"]==x["next_musteri"]])
             .reset_index(drop=True))

def build_seq_network(pairs_df, min_edge=None):
    if min_edge is None: min_edge = NET_SEQ_MIN_EDGE
    G = nx.DiGraph()
    for u in URUNLER:
        u_rows = pairs_df[pairs_df["urun_grubu"]==u] if not pairs_df.empty else pd.DataFrame()
        G.add_node(u, color=URUN_RENKLER[u], freq=len(u_rows),
                   volume=float(u_rows["islem_tutari"].sum()) if not u_rows.empty else 0.0)
    if pairs_df.empty: return G
    edge_stats = (pairs_df.groupby(["urun_grubu","next_urun"], observed=True)
                  .agg(count=("musteri_id","count"), volume=("islem_tutari","sum")).reset_index())
    edge_stats = edge_stats[edge_stats["count"] >= min_edge].copy()
    total_from = edge_stats.groupby("urun_grubu")["count"].sum()
    for _, row in edge_stats.iterrows():
        pct = row["count"] / max(total_from.get(row["urun_grubu"], 1), 1) * 100
        G.add_edge(row["urun_grubu"], row["next_urun"],
                   weight=int(row["count"]), pct=round(pct,1), volume=float(row["volume"]))
    return G

_pairs_all      = _compute_pairs(islem_df)
G_SEQ_ALL       = build_seq_network(_pairs_all, min_edge=NET_SEQ_MIN_EDGE)
G_SEQ_PRE_BUY   = build_seq_network(_compute_pairs(
    islem_window_df[(islem_window_df["event_type"]=="Alım") &(islem_window_df["pencere"]=="Pre")]) if len(islem_window_df)>=2 else pd.DataFrame(), min_edge=2)
G_SEQ_POST_BUY  = build_seq_network(_compute_pairs(
    islem_window_df[(islem_window_df["event_type"]=="Alım") &(islem_window_df["pencere"]=="Post")]) if len(islem_window_df)>=2 else pd.DataFrame(), min_edge=2)
G_SEQ_PRE_SELL  = build_seq_network(_compute_pairs(
    islem_window_df[(islem_window_df["event_type"]=="Satım")&(islem_window_df["pencere"]=="Pre")]) if len(islem_window_df)>=2 else pd.DataFrame(), min_edge=2)
G_SEQ_POST_SELL = build_seq_network(_compute_pairs(
    islem_window_df[(islem_window_df["event_type"]=="Satım")&(islem_window_df["pencere"]=="Post")]) if len(islem_window_df)>=2 else pd.DataFrame(), min_edge=2)
G_SEQ_SEG = {seg: build_seq_network(
    _pairs_all[_pairs_all["musteri_segmenti"]==seg].reset_index(drop=True), min_edge=3)
    for seg in SEGMENT_SIRASI}

def _net_metrics(G):
    base = {u: {"pagerank":0.0,"betweenness":0.0,"in_deg":0,"out_deg":0} for u in URUNLER}
    if G.number_of_edges()==0: return base
    pr = nx.pagerank(G, weight="weight", max_iter=500, tol=1e-6)
    bc = nx.betweenness_centrality(G, weight="weight", normalized=True)
    return {u: {"pagerank":round(pr.get(u,0),4),"betweenness":round(bc.get(u,0),4),
                "in_deg":G.in_degree(u,weight="weight"),"out_deg":G.out_degree(u,weight="weight")}
            for u in URUNLER}

NET_METRICS    = _net_metrics(G_SEQ_ALL)
NET_METRICS_DF = pd.DataFrame(NET_METRICS).T.reset_index().rename(columns={"index":"urun"})

def _trans_mat(pairs_df):
    if pairs_df.empty:
        return pd.DataFrame(0.0, index=URUNLER, columns=URUNLER)
    mat = (pairs_df.groupby(["urun_grubu","next_urun"], observed=True)
                   .size().unstack(fill_value=0)
                   .reindex(index=URUNLER, columns=URUNLER, fill_value=0))
    return mat.div(mat.sum(axis=1).clip(lower=1), axis=0).round(3)

TRANS_MAT_ALL      = _trans_mat(_pairs_all)
TRANS_MAT_PRE_BUY  = _trans_mat(_compute_pairs(
    islem_window_df[(islem_window_df["event_type"]=="Alım")&(islem_window_df["pencere"]=="Pre")]) if len(islem_window_df)>=2 else pd.DataFrame())
TRANS_MAT_POST_BUY = _trans_mat(_compute_pairs(
    islem_window_df[(islem_window_df["event_type"]=="Alım")&(islem_window_df["pencere"]=="Post")]) if len(islem_window_df)>=2 else pd.DataFrame())

def _top_ngrams(n=3, top_k=15):
    d = (islem_df[["musteri_id","islem_tarihi","urun_grubu"]]
         .sort_values(["musteri_id","islem_tarihi"]).reset_index(drop=True))
    grams = []
    for _, grp in d.groupby("musteri_id", sort=False):
        seq = grp["urun_grubu"].tolist()
        for i in range(len(seq)-n+1):
            grams.append(" → ".join(seq[i:i+n]))
    return pd.Series(_Counter(grams)).sort_values(ascending=False).head(top_k)

TOP_TRIGRAMS = _top_ngrams(n=3, top_k=15)
TOP_BIGRAMS  = (_pairs_all.groupby(["urun_grubu","next_urun"], observed=True)
                .size().reset_index(name="count")
                .assign(label=lambda d: d["urun_grubu"]+" → "+d["next_urun"])
                .sort_values("count", ascending=False).head(15).reset_index(drop=True))

# §4.16 KPI
KPI = {
    "toplam_alim_m":    round(alim_df["alim_tutari"].sum()/1e6, 1),
    "toplam_satim_m":   round(satim_df["satim_tutari"].sum()/1e6, 1),
    "net_aum_m":        round(AUM.sum()/1e6, 1),
    "toplam_event":     len(events_df),
    "alim_musteri":     alim_df["musteri_id"].nunique(),
    "satim_musteri":    satim_df["musteri_id"].nunique(),
    "window_islem":     len(islem_window_df),
    "penetrasyon_ort":  PENETRASYON["Alim_Pct"].mean().round(1),
    "sadakat_ort":      SADAKAT["Cok_Donem_Pct"].mean().round(1),
    "aktivite_max_seg": DAVRANIS_SKOR["Aktivite_Skoru"].idxmax(),
}

print(f"\n{'═'*58}")
print(f"  ✅  §4 Analiz Engine tamamlandı")
print(f"  KPI: Alım {KPI['toplam_alim_m']:.1f}M TL · AUM {KPI['net_aum_m']:.1f}M TL")
print(f"{'═'*58}")


# ══════════════════════════════════════════════════════════════════
# § 4.17–4.20 · DÖNEMSELMETRİKLER · HEAVY · KOMPOZİT AĞ
# ══════════════════════════════════════════════════════════════════

# §4.17 Dönemsel Metrikler
DONEM_ALIM = (alim_df.merge(musteri_df, on="musteri_id")
              .groupby(["donem","musteri_segmenti"], observed=True)
              .agg(Alim_Adet=("alim_tutari","count"),
                   Alim_Tutar_M=("alim_tutari", lambda x: round(x.sum()/1e6,3)),
                   Alim_Musteri=("musteri_id","nunique")).reset_index())
DONEM_SATIM = (satim_df.merge(musteri_df, on="musteri_id")
               .groupby(["donem","musteri_segmenti"], observed=True)
               .agg(Satim_Adet=("satim_tutari","count"),
                    Satim_Tutar_M=("satim_tutari", lambda x: round(x.sum()/1e6,3)),
                    Satim_Musteri=("musteri_id","nunique")).reset_index())

DONEM_ALIM_PIVOT  = (DONEM_ALIM.pivot(index="musteri_segmenti", columns="donem", values="Alim_Tutar_M")
                     .reindex(SEGMENT_SIRASI)[DONEM_SIRASI].fillna(0))
DONEM_SATIM_PIVOT = (DONEM_SATIM.pivot(index="musteri_segmenti", columns="donem", values="Satim_Tutar_M")
                     .reindex(SEGMENT_SIRASI)[DONEM_SIRASI].fillna(0))
DONEM_NET_PIVOT   = DONEM_ALIM_PIVOT.sub(DONEM_SATIM_PIVOT, fill_value=0).round(3)
DONEM_ALIM_POC    = DONEM_ALIM_PIVOT.pct_change(axis=1).multiply(100).round(1)
DONEM_SATIM_POC   = DONEM_SATIM_PIVOT.pct_change(axis=1).multiply(100).round(1)

def _assign_donem_col(tarih_series):
    res = pd.Series(pd.NA, index=tarih_series.index, dtype=object)
    for _d in DONEM_SIRASI:
        _bas = pd.Timestamp(DONEM_ARALIK[_d][0]); _bit = pd.Timestamp(DONEM_ARALIK[_d][1])
        res[(tarih_series >= _bas) & (tarih_series <= _bit)] = _d
    return res

_islem_d = islem_df[["musteri_id","musteri_segmenti","islem_tarihi","islem_tutari","urun_grubu"]].copy()
_islem_d["donem"] = _assign_donem_col(_islem_d["islem_tarihi"])
DONEM_ISLEM_PIVOT = (_islem_d.dropna(subset=["donem"])
                     .groupby(["musteri_segmenti","donem"], observed=True)["islem_tutari"]
                     .sum().div(1e6).round(2).unstack(fill_value=0)
                     .reindex(SEGMENT_SIRASI).reindex(columns=DONEM_SIRASI, fill_value=0))

# §4.18 Heavy Buyer / Seller
alim_mst = (alim_df.merge(musteri_df, on="musteri_id")
            .groupby(["musteri_id","musteri_segmenti","donem"], observed=True)
            ["alim_tutari"].sum().reset_index(name="toplam_alim"))

def _heavy_flag(grp, col):
    grp = grp.copy(); grp["heavy"] = grp[col] >= grp[col].quantile(0.75); return grp

alim_mst  = (alim_mst.groupby(["musteri_segmenti","donem"], observed=True, group_keys=False)
             .apply(lambda g: _heavy_flag(g, "toplam_alim")))
satim_mst = (satim_df.merge(musteri_df, on="musteri_id")
             .groupby(["musteri_id","musteri_segmenti","donem"], observed=True)
             ["satim_tutari"].sum().reset_index(name="toplam_satim"))
satim_mst = (satim_mst.groupby(["musteri_segmenti","donem"], observed=True, group_keys=False)
             .apply(lambda g: _heavy_flag(g, "toplam_satim")))

HEAVY_BUYER_SEG  = (alim_mst.groupby(["musteri_segmenti","donem"], observed=True)["heavy"]
                    .mean().multiply(100).round(1).unstack(fill_value=0)
                    .reindex(SEGMENT_SIRASI).reindex(columns=DONEM_SIRASI, fill_value=0))
HEAVY_SELLER_SEG = (satim_mst.groupby(["musteri_segmenti","donem"], observed=True)["heavy"]
                    .mean().multiply(100).round(1).unstack(fill_value=0)
                    .reindex(SEGMENT_SIRASI).reindex(columns=DONEM_SIRASI, fill_value=0))
HEAVY_BUYER_TUTAR  = (alim_mst.groupby(["musteri_segmenti","heavy"], observed=True)["toplam_alim"]
                      .mean().div(1e3).round(1).unstack().reindex(SEGMENT_SIRASI)
                      .rename(columns={False:"Normal_K", True:"Heavy_K"}))
HEAVY_SELLER_TUTAR = (satim_mst.groupby(["musteri_segmenti","heavy"], observed=True)["toplam_satim"]
                      .mean().div(1e3).round(1).unstack().reindex(SEGMENT_SIRASI)
                      .rename(columns={False:"Normal_K", True:"Heavy_K"}))
_heavy_ids  = set(alim_mst.loc[alim_mst["heavy"]==True, "musteri_id"].tolist())
_normal_ids = set(alim_mst.loc[alim_mst["heavy"]==False, "musteri_id"].tolist())
HEAVY_ISLEM_DONEM = (_islem_d[_islem_d["musteri_id"].isin(_heavy_ids) & _islem_d["donem"].notna()]
                     .groupby(["musteri_segmenti","donem"], observed=True)["islem_tutari"]
                     .mean().div(1e3).round(1).unstack(fill_value=0)
                     .reindex(SEGMENT_SIRASI).reindex(columns=DONEM_SIRASI, fill_value=0))

# §4.19 Kompozit Boyut
islem_df["urun_tur_yon"] = (islem_df["urun_grubu"] + "|" +
                             islem_df["islem_turu"]  + "|" +
                             islem_df["islem_yonu"])
COMPOSITE_FREQ = (islem_df["urun_tur_yon"].value_counts().rename_axis("node")
                  .reset_index(name="count").head(20))

def _short_comp_label(n):
    parts = n.split("|")
    return f"{parts[0][:4]}/{parts[1][:5] if len(parts)>1 else ''}/{parts[2][0] if len(parts)>2 else ''}"

# §4.20 Kompozit Network
def build_composite_net(df, col="urun_tur_yon", min_edge=5):
    d = (df[["musteri_id","islem_tarihi",col,"islem_tutari"]]
         .sort_values(["musteri_id","islem_tarihi"]).reset_index(drop=True))
    d["next_node"] = d.groupby("musteri_id", sort=False)[col].shift(-1)
    d["next_id"]   = d.groupby("musteri_id", sort=False)["musteri_id"].shift(-1)
    pairs = (d.dropna(subset=["next_node"]).pipe(lambda x: x[x["musteri_id"]==x["next_id"]])
              .reset_index(drop=True))
    G = nx.DiGraph()
    if pairs.empty: return G, pairs
    edge_stats = (pairs.groupby([col,"next_node"], observed=True)
                  .agg(count=("musteri_id","count"), volume=("islem_tutari","sum")).reset_index())
    edge_stats = edge_stats[edge_stats["count"] >= min_edge].copy()
    nodes = set(edge_stats[col].tolist() + edge_stats["next_node"].tolist())
    _freq_map = islem_df[col].value_counts().to_dict()
    for n in nodes:
        urun = n.split("|")[0]
        G.add_node(n, color=URUN_RENKLER.get(urun,"#94A3B8"), urun=urun, freq=_freq_map.get(n,0))
    total_from = edge_stats.groupby(col)["count"].sum()
    for _, row in edge_stats.iterrows():
        pct = row["count"] / max(total_from.get(row[col], 1), 1) * 100
        G.add_edge(row[col], row["next_node"],
                   weight=int(row["count"]), pct=round(pct,1), volume=float(row["volume"]))
    return G, pairs

G_COMPOSITE, _composite_pairs = build_composite_net(islem_df, min_edge=8)
G_COMPOSITE_SEG = {}
for _cs in SEGMENT_SIRASI:
    G_COMPOSITE_SEG[_cs], _ = build_composite_net(
        islem_df[islem_df["musteri_segmenti"]==_cs].reset_index(drop=True), min_edge=3)

print(f"\n✅ §4.17-20 Dönemsel metrikler + Heavy + Kompozit ağ")
print(f"   Kompozit: {G_COMPOSITE.number_of_nodes()} node · {G_COMPOSITE.number_of_edges()} kenar")
print(f"   Heavy: {len(_heavy_ids):,} heavy buyer · {len(_normal_ids):,} normal")


# ══════════════════════════════════════════════════════════════════
# § 5 · t-1 BAKİYE HESAPLAMALARI
# ══════════════════════════════════════════════════════════════════
# 2 dönemde t-1: 2026.03 için önceki dönem 2025.09
_prev_donem_map = {d: DONEM_SIRASI[i-1]
                   for i, d in enumerate(DONEM_SIRASI) if i > 0}
_donems_with_prev = list(_prev_donem_map.keys())   # ["2026.03"]

_bak_t1_parts = []
for curr_d, prev_d in _prev_donem_map.items():
    _sub = (bakiye_df[bakiye_df["donem"]==prev_d]
            [["musteri_id","fon_bakiye_tutari"]].copy()
            .rename(columns={"fon_bakiye_tutari":"bakiye_t1"}))
    _sub["donem"] = curr_d
    _bak_t1_parts.append(_sub)

bakiye_t1_map = (pd.concat(_bak_t1_parts, ignore_index=True) if _bak_t1_parts
                 else pd.DataFrame(columns=["musteri_id","bakiye_t1","donem"]))

alim_d_mst = (alim_df.merge(musteri_df[["musteri_id","musteri_segmenti"]], on="musteri_id")
              .groupby(["musteri_id","musteri_segmenti","donem"], observed=True)
              ["alim_tutari"].sum().reset_index(name="alim_tutari"))
satim_d_mst = (satim_df.merge(musteri_df[["musteri_id","musteri_segmenti"]], on="musteri_id")
               .groupby(["musteri_id","musteri_segmenti","donem"], observed=True)
               ["satim_tutari"].sum().reset_index(name="satim_tutari"))

alim_t1 = (alim_d_mst[alim_d_mst["donem"].isin(_donems_with_prev)]
           .merge(bakiye_t1_map, on=["musteri_id","donem"], how="left")
           .dropna(subset=["bakiye_t1"]))
alim_t1["alim_bak_pct"] = (
    alim_t1["alim_tutari"] / alim_t1["bakiye_t1"].clip(lower=1) * 100).clip(0, 500).round(2)

satim_t1 = (satim_d_mst[satim_d_mst["donem"].isin(_donems_with_prev)]
            .merge(bakiye_t1_map, on=["musteri_id","donem"], how="left")
            .dropna(subset=["bakiye_t1"]))
satim_t1["satim_bak_pct"] = (
    satim_t1["satim_tutari"] / satim_t1["bakiye_t1"].clip(lower=1) * 100).clip(0, 500).round(2)

def _seg_donem_hm(df, val_col):
    return (df.groupby(["musteri_segmenti","donem"], observed=True)[val_col]
            .median().round(2).unstack(fill_value=0)
            .reindex(SEGMENT_SIRASI).reindex(columns=_donems_with_prev, fill_value=0))

ALIM_BAK_HM   = _seg_donem_hm(alim_t1,  "alim_bak_pct")
SATIM_BAK_HM  = _seg_donem_hm(satim_t1, "satim_bak_pct")
ALIM_BAK_DELTA  = ALIM_BAK_HM.diff(axis=1).fillna(0).round(2)
SATIM_BAK_DELTA = SATIM_BAK_HM.diff(axis=1).fillna(0).round(2)
print(f"\n✅ § 5 t-1 Bakiye hesaplandı  |  Dönem: {_donems_with_prev}")


# ══════════════════════════════════════════════════════════════════
# § 6 · FON-GEÇMEYEN İŞLEM SEKANS HESAPLAMALARI
# ══════════════════════════════════════════════════════════════════
FON_GECEN_TURLER    = {"EFT","Havale","ATM_Çekim","Fatura_Ödeme","Kredi_Ödemesi","Kredi_Kullanımı"}
FON_GECMEYEN_URUNLER = {"Vadeli","Yatırım","Döviz"}
FG_TURLER = ["Vadeli_Açılış","Vadeli_Kapanış","Hisse_Alım","Hisse_Satım",
             "TahvilBono_Alım","TahvilBono_Satım","Repo_Giriş","Döviz_Alım","Döviz_Satım"]
FG_TURLER_KISALT = {
    "Vadeli_Açılış":"Vdl.Açılış","Vadeli_Kapanış":"Vdl.Kapanış",
    "Hisse_Alım":"Hisse.Alım","Hisse_Satım":"Hisse.Satım",
    "TahvilBono_Alım":"TahvBono.A","TahvilBono_Satım":"TahvBono.S",
    "Repo_Giriş":"Repo.Giriş","Döviz_Alım":"Döviz.Alım","Döviz_Satım":"Döviz.Satım",
}

def _fg_filtre(df):
    return df[df["urun_grubu"].isin(FON_GECMEYEN_URUNLER) &
              (~df["islem_turu"].isin(FON_GECEN_TURLER))].copy()

fg_pre_buy   = _fg_filtre(pre_buy)
fg_post_buy  = _fg_filtre(post_buy)
fg_pre_sell  = _fg_filtre(pre_sell)
fg_post_sell = _fg_filtre(post_sell)

def _seg_tur_pct(df):
    if df.empty: return pd.DataFrame(0.0, index=SEGMENT_SIRASI, columns=FG_TURLER)
    sub = df[df["islem_turu"].isin(FG_TURLER)].copy()
    if sub.empty: return pd.DataFrame(0.0, index=SEGMENT_SIRASI, columns=FG_TURLER)
    grp = sub.groupby(["musteri_segmenti","islem_turu"], observed=True).size().reset_index(name="n")
    grp["toplam"] = grp.groupby("musteri_segmenti", observed=True)["n"].transform("sum")
    grp["pct"]    = (grp["n"] / grp["toplam"].clip(lower=1) * 100).round(1)
    return (grp.pivot(index="musteri_segmenti", columns="islem_turu", values="pct")
            .fillna(0).reindex(SEGMENT_SIRASI).reindex(columns=FG_TURLER, fill_value=0))

DIST_PRE_BUY   = _seg_tur_pct(fg_pre_buy)
DIST_POST_BUY  = _seg_tur_pct(fg_post_buy)
DIST_PRE_SELL  = _seg_tur_pct(fg_pre_sell)
DIST_POST_SELL = _seg_tur_pct(fg_post_sell)

def _seg_donem_tur(df):
    return {d: _seg_tur_pct(df[df["donem"]==d] if not df.empty else pd.DataFrame())
            for d in DONEM_SIRASI}

DIST_PRE_BUY_D   = _seg_donem_tur(fg_pre_buy)
DIST_POST_BUY_D  = _seg_donem_tur(fg_post_buy)
DIST_PRE_SELL_D  = _seg_donem_tur(fg_pre_sell)
DIST_POST_SELL_D = _seg_donem_tur(fg_post_sell)

print(f"\n✅ § 6 Fon-Geçmeyen Sekans")
print(f"   Pre-Buy: {len(fg_pre_buy):,}  Post-Buy: {len(fg_post_buy):,}  "
      f"Pre-Sell: {len(fg_pre_sell):,}  Post-Sell: {len(fg_post_sell):,}")


# ══════════════════════════════════════════════════════════════════
# ▶  PART 1 TAMAMLANDI
# ══════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("  ✅  PART 1 TAMAMLANDI — Tüm analizler hazır")
print(f"{'═'*60}")
print(f"  Dönemler     : {DONEM_SIRASI}  (Eylül 2025 · Mart 2026)")
print(f"  Segmentler   : {len(SEGMENT_SIRASI)} segment · {N_MUSTERI:,} müşteri")
print(f"  Alım         : {KPI['toplam_alim_m']:.1f}M TL  |  {KPI['alim_musteri']:,} müşteri")
print(f"  AUM          : {KPI['net_aum_m']:.1f}M TL")
print(f"  t-1 Dönem    : {_donems_with_prev}")
print(f"  Kompozit Ağ  : {G_COMPOSITE.number_of_nodes()} node · {G_COMPOSITE.number_of_edges()} kenar")
print(f"\n  ▶  PART 2 hücresini çalıştırarak grafikleri üret.")


In [ ]:
# Cell 2 — Chartlar + Profesyonel Network Görselleştirmesi
# ╔══════════════════════════════════════════════════════════════════╗
# ║  PART 2 — CHARTLAR · VİZÜALİZASYON · PROFESYONEL NETWORK       ║
# ║  PART 1 çalıştırıldıktan sonra bu hücreyi çalıştırın            ║
# ╚══════════════════════════════════════════════════════════════════╝

import math as _math
import plotly.graph_objects as go
import os
from IPython.display import HTML, display

_N  = len(SEGMENT_SIRASI)
_ND = len(DONEM_SIRASI)
xs  = np.arange(_N)

# ══════════════════════════════════════════════════════════════════
# § A — GENEL BAKIŞ  (6 panel)
# ══════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 11))
gs  = GridSpec(2, 3, figure=fig, hspace=0.52, wspace=0.40)
ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1]); ax3 = fig.add_subplot(gs[0,2])
ax4 = fig.add_subplot(gs[1,0]); ax5 = fig.add_subplot(gs[1,1]); ax6 = fig.add_subplot(gs[1,2])

b1 = ax1.bar(xs-0.2, PENETRASYON["Alim_Pct"],  width=0.38, color=SEG_RENK_LIST, alpha=0.9,  zorder=3)
b2 = ax1.bar(xs+0.2, PENETRASYON["Satim_Pct"], width=0.38, color=SEG_RENK_LIST, alpha=0.45, zorder=3, hatch="//")
for bg in [b1,b2]:
    for p in bg.patches:
        h = p.get_height()
        if h > 1: ax1.text(p.get_x()+p.get_width()/2, h+1, f"{h:.0f}%",
                            ha="center", va="bottom", fontsize=6.5, fontweight="bold", color=TEXT_CLR)
ax1.set_xticks(xs); ax1.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax1.set_ylabel("Penetrasyon (%)"); ax1.set_ylim(0, 115)
ax1.set_title("Fon Penetrasyonu\n(Alım / Satım Yapan Müşteri %)")
ax1.legend(handles=[mpatches.Patch(facecolor="#555",label="Alım"),
                    mpatches.Patch(facecolor="#aaa",hatch="//",label="Satım")],
           loc="upper right", fontsize=8); style_axes(ax1)

aum_m = AUM / 1e6
b3 = ax2.bar(xs, aum_m.values, color=SEG_RENK_LIST, alpha=0.9, zorder=3, edgecolor="white", linewidth=0.8)
for p in b3.patches:
    h = p.get_height()
    if h > 0:
        lbl = f"{h:.0f}M" if h < 1000 else f"{h/1e3:.1f}B"
        ax2.text(p.get_x()+p.get_width()/2, h+aum_m.max()*0.015,
                 lbl, ha="center", va="bottom", fontsize=7, fontweight="bold", color=TEXT_CLR)
ax2.set_xticks(xs); ax2.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax2.set_ylabel("Net AUM (M TL)"); ax2.set_title("Net Fon Pozisyonu\n(Toplam Alım − Satım)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}M")); style_axes(ax2)

ort_alim  = METRIK_GENEL[METRIK_GENEL["event_type"]=="Alım"].set_index("musteri_segmenti")["Ort_Tutar_K"].reindex(SEGMENT_SIRASI)
ort_satim = METRIK_GENEL[METRIK_GENEL["event_type"]=="Satım"].set_index("musteri_segmenti")["Ort_Tutar_K"].reindex(SEGMENT_SIRASI)
y_pos = np.arange(_N)
ax3.scatter(ort_alim.values,  y_pos, s=150, c=SEG_RENK_LIST, marker="o", zorder=4, label="Alım ort.", linewidths=1.5, edgecolors="white")
ax3.scatter(ort_satim.values, y_pos, s=150, c=SEG_RENK_LIST, marker="D", zorder=4, label="Satım ort.", alpha=0.65, linewidths=1.5, edgecolors="white")
for i, seg in enumerate(SEGMENT_SIRASI):
    a, s_ = ort_alim.iloc[i], ort_satim.iloc[i]
    ax3.plot([min(a,s_),max(a,s_)],[i,i], color=SEG_RENK[seg], lw=1.5, alpha=0.4, zorder=2)
ax3.set_yticks(y_pos); ax3.set_yticklabels(SEG_SHORT, fontsize=7.5)
ax3.set_xlabel("Ortalama İşlem Tutarı (Bin TL)"); ax3.set_title("Ort. İşlem Tutarı\n(● Alım  ◆ Satım)")
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}K"))
ax3.legend(loc="lower right", fontsize=8); style_axes(ax3)

ax4.barh(xs+0.2, SADAKAT["Cok_Donem_Pct"].values, height=0.35, color=SEG_RENK_LIST, alpha=0.9, label="Her 2 dönemde", zorder=3)
ax4.barh(xs-0.2, SADAKAT["Tek_Donem_Pct"].values,  height=0.35, color=SEG_RENK_LIST, alpha=0.40, label="Tek dönem", zorder=3)
ax4.set_yticks(xs); ax4.set_yticklabels(SEG_SHORT, fontsize=7.5)
ax4.set_xlabel("Müşteri Oranı (%)"); ax4.set_title("Müşteri Sadakati\n(Her İki Dönem vs Tek Dönem Alım)")
ax4.axvline(50, ls="--", color=SPINE_CLR, lw=1.0, alpha=0.7)
ax4.legend(loc="lower right", fontsize=8); style_axes(ax4)

bd_pct = (musteri_bakiye_dilim.merge(musteri_df, on="musteri_id")
          .groupby(["musteri_segmenti","bakiye_dilimi"], observed=True).size()
          .unstack(fill_value=0).reindex(SEGMENT_SIRASI)
          .reindex(columns=BAKIYE_DILIMLERI, fill_value=0))
bd_pct_norm = bd_pct.div(bd_pct.sum(axis=1), axis=0) * 100
bottom = np.zeros(_N)
for dil in BAKIYE_DILIMLERI:
    vals = bd_pct_norm[dil].values
    ax5.bar(xs, vals, bottom=bottom, width=0.55, label=dil,
            color=BAKIYE_RENK[dil], alpha=0.9, zorder=3, edgecolor="white", linewidth=0.6)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 8: ax5.text(xs[i], b+v/2, f"{v:.0f}%", ha="center", va="center",
                            fontsize=7, fontweight="bold", color=TEXT_CLR)
    bottom += vals
ax5.set_xticks(xs); ax5.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax5.set_ylabel("Müşteri Oranı (%)"); ax5.set_ylim(0, 115)
ax5.set_title("Bakiye Dilimi Dağılımı\n(Segment içi percentile)")
ax5.legend(loc="upper right", fontsize=8, title="Dilim"); style_axes(ax5)

t_alim_m  = (alim_df.merge(musteri_df,  on="musteri_id")
             .groupby("musteri_segmenti", observed=True)["alim_tutari"].sum()
             .reindex(SEGMENT_SIRASI) / 1e6)
t_satim_m = (satim_df.merge(musteri_df, on="musteri_id")
             .groupby("musteri_segmenti", observed=True)["satim_tutari"].sum()
             .reindex(SEGMENT_SIRASI) / 1e6)
ax6.bar(xs-0.2, t_alim_m.values,  width=0.38, color=SEG_RENK_LIST, alpha=0.9,  zorder=3)
ax6.bar(xs+0.2, t_satim_m.values, width=0.38, color=SEG_RENK_LIST, alpha=0.45, zorder=3, hatch="//")
ax6.set_xticks(xs); ax6.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax6.set_ylabel("Toplam Hacim (M TL)"); ax6.set_title("Fon İşlem Hacmi\n(Alım vs Satım, M TL)")
ax6.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}M"))
ax6.legend(handles=[mpatches.Patch(facecolor="#555",label="Alım"),
                    mpatches.Patch(facecolor="#aaa",hatch="//",label="Satım")],
           loc="upper left", fontsize=8); style_axes(ax6)

style_fig(fig, "A — Genel Bakış · Banka Fon Müşteri Analizi  ·  2025.09 & 2026.03",
          f"Penetrasyon · AUM · Sadakat · Bakiye Dilimi  |  {len(events_df):,} olay · {_N} segment · 2 dönem")
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()


# ══════════════════════════════════════════════════════════════════
# § B — FON ALIM DAVRANIŞI  (6 panel)
# ══════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 12))
gs  = GridSpec(2, 3, figure=fig, hspace=0.52, wspace=0.40)
ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1]); ax3 = fig.add_subplot(gs[0,2])
ax4 = fig.add_subplot(gs[1,0]); ax5 = fig.add_subplot(gs[1,1]); ax6 = fig.add_subplot(gs[1,2])

for urun in URUNLER:
    bottom = np.zeros(_N) if urun == URUNLER[0] else bottom
    vals = URUN_PRE_BUY[urun].values
    ax1.bar(xs, vals, bottom=bottom, width=0.55, label=urun,
            color=URUN_RENKLER[urun], alpha=0.9, zorder=3, edgecolor="white", linewidth=0.5)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 9: ax1.text(xs[i], b+v/2, f"{v:.0f}%", ha="center", va="center", fontsize=6.5, fontweight="bold", color="white")
    bottom += vals
ax1.set_xticks(xs); ax1.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax1.set_ylabel("İşlem Dağılımı (%)"); ax1.set_ylim(0,115)
ax1.set_title(f"Pre-Buy Ürün Mix\n(Alım öncesi −{X_DAYS} gün)")
ax1.legend(loc="upper right", fontsize=8, title="Ürün Grubu"); style_axes(ax1)

bottom = np.zeros(_N)
for urun in URUNLER:
    vals = URUN_POST_BUY[urun].values
    ax2.bar(xs, vals, bottom=bottom, width=0.55, label=urun,
            color=URUN_RENKLER[urun], alpha=0.9, zorder=3, edgecolor="white", linewidth=0.5)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 9: ax2.text(xs[i], b+v/2, f"{v:.0f}%", ha="center", va="center", fontsize=6.5, fontweight="bold", color="white")
    bottom += vals
ax2.set_xticks(xs); ax2.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax2.set_ylabel("İşlem Dağılımı (%)"); ax2.set_ylim(0,115)
ax2.set_title(f"Post-Buy Ürün Mix\n(Alım sonrası +{X_DAYS} gün)")
ax2.legend(loc="upper right", fontsize=8, title="Ürün Grubu"); style_axes(ax2)

w = 0.35
ax3.bar(xs-w/2, GIRIS_PRE_BUY.values,  width=w, color=YON_RENK["Giriş"], alpha=0.9,  label="Pre-Buy",  zorder=3)
ax3.bar(xs+w/2, GIRIS_POST_BUY.values, width=w, color=YON_RENK["Giriş"], alpha=0.50, label="Post-Buy", zorder=3)
ax3.axhline(50, ls="--", color=SPINE_CLR, lw=1.0, alpha=0.7)
for i, (a, b_) in enumerate(zip(GIRIS_PRE_BUY.values, GIRIS_POST_BUY.values)):
    ax3.text(i-w/2, a+1, f"{a:.0f}%",  ha="center", va="bottom", fontsize=7, fontweight="bold", color=TEXT_CLR)
    ax3.text(i+w/2, b_+1, f"{b_:.0f}%", ha="center", va="bottom", fontsize=7, fontweight="bold", color=TEXT_CLR)
ax3.set_xticks(xs); ax3.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax3.set_ylabel("Giriş Oranı (%)"); ax3.set_ylim(0, 85)
ax3.set_title("Para Giriş Oranı\n(Pre-Buy vs Post-Buy)")
ax3.legend(fontsize=8); style_axes(ax3)

days = GUNLUK_ALIM.index.values
ax4.bar(days, GUNLUK_ALIM.values/1e3,
        color=[YON_RENK["Giriş"] if d < 0 else "#1D4ED8" for d in days],
        alpha=0.85, zorder=3, width=0.8)
ax4.axvline(0, ls="--", color="#F59E0B", lw=2.0, zorder=5)
ax4.set_xlabel("Alım Olayına Göre Gün"); ax4.set_ylabel("Ort. İşlem Tutarı (K TL)")
ax4.set_title("Günlük İşlem Hacmi\n(Alım Event Window)")
ax4.set_xticks(range(-X_DAYS, X_DAYS+1))
ax4.set_xticklabels([str(d) if d%2==0 else "" for d in range(-X_DAYS, X_DAYS+1)], fontsize=8)
ax4.legend(handles=[mpatches.Patch(color=YON_RENK["Giriş"],label=f"Pre"),
                    mpatches.Patch(color="#1D4ED8",label="Post"),
                    Line2D([0],[0],color="#F59E0B",lw=2,ls="--",label="Alım")],
           loc="upper left", fontsize=8); style_axes(ax4)

for i, seg in enumerate(SEGMENT_SIRASI):
    net_pre  = NET_PRE_BUY.get(seg, 0) / 1e3
    net_post = NET_POST_BUY.get(seg, 0) / 1e3
    ax5.plot([0, 1], [net_pre, net_post], "o-", color=SEG_RENK[seg], lw=1.8, ms=5, alpha=0.88)
    ax5.text(-0.08, net_pre,  f"{SEG_SHORT[i]}", ha="right", va="center", fontsize=7, color=SEG_RENK[seg])
    ax5.text(1.08,  net_post, f"{net_post:.0f}K", ha="left", va="center", fontsize=7, color=SEG_RENK[seg])
ax5.axhline(0, ls="--", color=SPINE_CLR, lw=1.0, alpha=0.7)
ax5.set_xticks([0,1]); ax5.set_xticklabels(["Pre-Buy","Post-Buy"], fontsize=9)
ax5.set_ylabel("Ort. Net Akış (K TL)"); ax5.set_title("Net Akış Değişimi\n(Pre-Buy → Post-Buy)")
ax5.set_xlim(-0.45, 1.45); style_axes(ax5)

sns.heatmap(GECIS_ALIM, annot=True, fmt=".0f", cmap="Blues",
            linewidths=0.5, linecolor=GRID_CLR, ax=ax6,
            cbar_kws={"shrink":0.85, "label":"Geçiş %"},
            xticklabels=URUNLER, yticklabels=URUNLER)
ax6.set_title("Pre→Post-Buy Ürün Geçiş\n(Satır-Normalize %)"); ax6.set_xlabel("Sonraki Ürün"); ax6.set_ylabel("Önceki Ürün")
ax6.tick_params(labelsize=8.5)

style_fig(fig, "B — Fon Alım Davranışı  ·  Segment × Dönem Bazlı",
          f"Pre/Post Alım Analizi  |  Ürün Mix · Giriş Oranı · Net Akış · Geçiş Matrisi  ·  2 dönem")
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()


# ══════════════════════════════════════════════════════════════════
# § C — FON SATIM DAVRANIŞI  (4 panel)
# ══════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 10))
gs  = GridSpec(2, 2, figure=fig, hspace=0.50, wspace=0.40)
ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1])
ax3 = fig.add_subplot(gs[1,0]); ax4 = fig.add_subplot(gs[1,1])

bottom = np.zeros(_N)
for urun in URUNLER:
    vals = URUN_PRE_SELL[urun].values
    ax1.bar(xs, vals, bottom=bottom, width=0.55, label=urun,
            color=URUN_RENKLER[urun], alpha=0.9, zorder=3, edgecolor="white", linewidth=0.5)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 9: ax1.text(xs[i], b+v/2, f"{v:.0f}%", ha="center", va="center", fontsize=6.5, fontweight="bold", color="white")
    bottom += vals
ax1.set_xticks(xs); ax1.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax1.set_title(f"Pre-Sell Ürün Mix\n(Satım öncesi −{X_DAYS} gün)")
ax1.set_ylabel("İşlem Dağılımı (%)"); ax1.set_ylim(0,115)
ax1.legend(loc="upper right", fontsize=8); style_axes(ax1)

bottom = np.zeros(_N)
for urun in URUNLER:
    vals = URUN_POST_SELL[urun].values
    ax2.bar(xs, vals, bottom=bottom, width=0.55, label=urun,
            color=URUN_RENKLER[urun], alpha=0.9, zorder=3, edgecolor="white", linewidth=0.5)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 9: ax2.text(xs[i], b+v/2, f"{v:.0f}%", ha="center", va="center", fontsize=6.5, fontweight="bold", color="white")
    bottom += vals
ax2.set_xticks(xs); ax2.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax2.set_title(f"Post-Sell Ürün Mix\n(Satım sonrası +{X_DAYS} gün)")
ax2.set_ylabel("İşlem Dağılımı (%)"); ax2.set_ylim(0,115)
ax2.legend(loc="upper right", fontsize=8); style_axes(ax2)

ax3.bar(xs-w/2, GIRIS_PRE_SELL.values,  width=w, color=YON_RENK["Çıkış"], alpha=0.9,  label="Pre-Sell",  zorder=3)
ax3.bar(xs+w/2, GIRIS_POST_SELL.values, width=w, color=YON_RENK["Çıkış"], alpha=0.50, label="Post-Sell", zorder=3)
ax3.axhline(50, ls="--", color=SPINE_CLR, lw=1.0, alpha=0.7)
ax3.set_xticks(xs); ax3.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax3.set_ylabel("Giriş Oranı (%)"); ax3.set_ylim(0, 85)
ax3.set_title("Para Giriş Oranı\n(Pre-Sell vs Post-Sell)")
ax3.legend(fontsize=8); style_axes(ax3)

sns.heatmap(GECIS_SATIM, annot=True, fmt=".0f", cmap="Reds",
            linewidths=0.5, linecolor=GRID_CLR, ax=ax4,
            cbar_kws={"shrink":0.85,"label":"Geçiş %"},
            xticklabels=URUNLER, yticklabels=URUNLER)
ax4.set_title("Pre→Post-Sell Ürün Geçiş\n(Satır-Normalize %)"); ax4.set_xlabel("Sonraki"); ax4.set_ylabel("Önceki")
ax4.tick_params(labelsize=8.5)

style_fig(fig, "C — Fon Satım Davranışı  ·  Segment × Dönem Bazlı",
          "Pre/Post Satım Analizi  |  Ürün Mix · Giriş Oranı · Geçiş Matrisi  ·  2 dönem")
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()


# ══════════════════════════════════════════════════════════════════
# § D — DÖNEMSELANALİZLER + HEAVY B/S  (2 figura)
# ══════════════════════════════════════════════════════════════════

# D1 — Dönemsel Karşılaştırma
fig_d = plt.figure(figsize=(20, 10))
gs_d  = GridSpec(2, 2, figure=fig_d, hspace=0.52, wspace=0.40)
ax_d1 = fig_d.add_subplot(gs_d[0,0]); ax_d2 = fig_d.add_subplot(gs_d[0,1])
ax_d3 = fig_d.add_subplot(gs_d[1,0]); ax_d4 = fig_d.add_subplot(gs_d[1,1])

sns.heatmap(DONEM_ALIM_PIVOT, ax=ax_d1, annot=True, fmt=".2f", cmap="Blues",
            linewidths=0.5, linecolor=GRID_CLR, cbar_kws={"shrink":0.8,"label":"M TL"},
            xticklabels=DONEM_SIRASI,
            yticklabels=[s.replace("_","\n") for s in SEGMENT_SIRASI])
ax_d1.set_title("Dönem × Segment — Alım Hacmi (M TL)")
ax_d1.set_xlabel("Dönem"); ax_d1.set_ylabel("Segment")
ax_d1.tick_params(axis="x", labelsize=8.5, rotation=20)
ax_d1.tick_params(axis="y", labelsize=7)
ax_d1.title.set_color(TITLE_CLR)

sns.heatmap(DONEM_SATIM_PIVOT, ax=ax_d2, annot=True, fmt=".2f", cmap="Reds",
            linewidths=0.5, linecolor=GRID_CLR, cbar_kws={"shrink":0.8,"label":"M TL"},
            xticklabels=DONEM_SIRASI,
            yticklabels=[s.replace("_","\n") for s in SEGMENT_SIRASI])
ax_d2.set_title("Dönem × Segment — Satım Hacmi (M TL)")
ax_d2.set_xlabel("Dönem"); ax_d2.set_ylabel("Segment")
ax_d2.tick_params(axis="x", labelsize=8.5, rotation=20)
ax_d2.tick_params(axis="y", labelsize=7)
ax_d2.title.set_color(TITLE_CLR)

sns.heatmap(DONEM_NET_PIVOT, ax=ax_d3, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.5, linecolor=GRID_CLR,
            cbar_kws={"shrink":0.8,"label":"M TL (net)"},
            xticklabels=DONEM_SIRASI,
            yticklabels=[s.replace("_","\n") for s in SEGMENT_SIRASI])
ax_d3.set_title("Net Fon Akışı — Dönem × Segment\n(Alım − Satım M TL  ·  Yeşil=net alım)")
ax_d3.set_xlabel("Dönem"); ax_d3.set_ylabel("Segment")
ax_d3.tick_params(axis="x", labelsize=8.5, rotation=20)
ax_d3.tick_params(axis="y", labelsize=7)
ax_d3.title.set_color(TITLE_CLR)

sns.heatmap(DONEM_ISLEM_PIVOT, ax=ax_d4, annot=True, fmt=".1f", cmap="YlOrBr",
            linewidths=0.5, linecolor=GRID_CLR, cbar_kws={"shrink":0.8,"label":"M TL"},
            xticklabels=DONEM_SIRASI,
            yticklabels=[s.replace("_","\n") for s in SEGMENT_SIRASI])
ax_d4.set_title("Dönem İçi İşlem Yoğunluğu (M TL)")
ax_d4.set_xlabel("Dönem"); ax_d4.set_ylabel("Segment")
ax_d4.tick_params(axis="x", labelsize=8.5, rotation=20)
ax_d4.tick_params(axis="y", labelsize=7)
ax_d4.title.set_color(TITLE_CLR)

style_fig(fig_d, "D — Dönemsel Analizler  ·  Segment × Dönem  ·  2025.09 vs 2026.03",
          "Alım · Satım · Net Akış · İşlem Yoğunluğu  ·  Heatmap bazlı karşılaştırma")
plt.tight_layout(rect=[0,0,1,0.97])
plt.show()

# D2 — Heavy Buyer/Seller
fig_g = plt.figure(figsize=(20, 10))
gs_g  = GridSpec(2, 2, figure=fig_g, hspace=0.52, wspace=0.42)
ax_g1 = fig_g.add_subplot(gs_g[0,0]); ax_g2 = fig_g.add_subplot(gs_g[0,1])
ax_g3 = fig_g.add_subplot(gs_g[1,0]); ax_g4 = fig_g.add_subplot(gs_g[1,1])

sns.heatmap(HEAVY_BUYER_SEG, ax=ax_g1, annot=True, fmt=".0f", cmap="Blues",
            linewidths=0.5, linecolor=GRID_CLR, cbar_kws={"shrink":0.8,"label":"%"},
            xticklabels=DONEM_SIRASI, yticklabels=[s.replace("_","\n") for s in SEGMENT_SIRASI])
ax_g1.set_title("Heavy Buyer Oranı\n(Top %25 Alım · Dönem × Segment %)")
ax_g1.set_xlabel("Dönem"); ax_g1.tick_params(axis="x",labelsize=8.5,rotation=20)
ax_g1.tick_params(axis="y",labelsize=7); ax_g1.title.set_color(TITLE_CLR)

_g2_x = np.arange(_N)
_g2_n = (HEAVY_BUYER_TUTAR["Normal_K"].values if "Normal_K" in HEAVY_BUYER_TUTAR.columns else np.ones(_N))
_g2_h = (HEAVY_BUYER_TUTAR["Heavy_K"].values  if "Heavy_K"  in HEAVY_BUYER_TUTAR.columns else np.ones(_N))
ax_g2.bar(_g2_x-0.2, np.maximum(_g2_n,0.01), width=0.38, color=SEG_RENK_LIST, alpha=0.45, label="Normal", zorder=3, edgecolor="white")
ax_g2.bar(_g2_x+0.2, np.maximum(_g2_h,0.01), width=0.38, color=SEG_RENK_LIST, alpha=0.92, label="Heavy",  zorder=3, edgecolor="white")
ax_g2.set_xticks(_g2_x); ax_g2.set_xticklabels(SEG_SHORT, rotation=35, ha="right", fontsize=7.5)
ax_g2.set_ylabel("Ort. Alım Tutarı (K TL)"); ax_g2.set_title("Heavy vs Normal Alıcı\n(Ort. Tutar)")
ax_g2.set_yscale("log")
ax_g2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e3:.0f}M" if x>=1e3 else f"{x:.0f}K"))
ax_g2.legend(fontsize=9); style_axes(ax_g2, grid=False)

sns.heatmap(HEAVY_SELLER_SEG, ax=ax_g3, annot=True, fmt=".0f", cmap="Reds",
            linewidths=0.5, linecolor=GRID_CLR, cbar_kws={"shrink":0.8,"label":"%"},
            xticklabels=DONEM_SIRASI, yticklabels=[s.replace("_","\n") for s in SEGMENT_SIRASI])
ax_g3.set_title("Heavy Seller Oranı\n(Top %25 Satım · Dönem × Segment %)")
ax_g3.set_xlabel("Dönem"); ax_g3.tick_params(axis="x",labelsize=8.5,rotation=20)
ax_g3.tick_params(axis="y",labelsize=7); ax_g3.title.set_color(TITLE_CLR)

for j, (seg, short) in enumerate(zip(SEGMENT_SIRASI, SEG_SHORT)):
    _vals = [HEAVY_ISLEM_DONEM.loc[seg, d] if (seg in HEAVY_ISLEM_DONEM.index and d in HEAVY_ISLEM_DONEM.columns) else 0 for d in DONEM_SIRASI]
    ax_g4.plot(range(_ND), _vals, "o-", color=SEG_RENK[seg], lw=1.8, ms=5, alpha=0.88, label=short)
ax_g4.set_xticks(range(_ND)); ax_g4.set_xticklabels(DONEM_SIRASI, fontsize=9, rotation=15)
ax_g4.set_ylabel("Ort. İşlem Tutarı (K TL)"); ax_g4.set_title("Heavy Buyer İşlem Trendi\n(Dönem bazlı)")
ax_g4.legend(title="Segment", loc="upper left", fontsize=7, ncol=5); style_axes(ax_g4)

style_fig(fig_g, "D2 — Heavy Buyer / Seller Segment Analizi  ·  2 Dönem",
          f"Top %25 alım/satım müşteri profili  |  {len(_heavy_ids):,} heavy buyer · {_N} segment")
plt.tight_layout(rect=[0,0,1,0.97])
plt.show()


# ══════════════════════════════════════════════════════════════════
# § E — SEKANSİYEL İŞLEM AĞLARI
# ══════════════════════════════════════════════════════════════════
def draw_seq_net(ax, G, title, min_pct=NET_SEQ_MIN_PCT, show_pct=True):
    ax.set_facecolor(FIG_BG); ax.axis("off")
    ax.set_title(title, fontsize=10, fontweight="bold", color=TITLE_CLR, pad=8)
    if G is None or G.number_of_edges() == 0:
        ax.text(0.5, 0.5, "Yeterli veri yok", ha="center", va="center",
                transform=ax.transAxes, color=SPINE_CLR, fontsize=9); return
    try: pos = nx.spring_layout(G, weight="pct", seed=42, k=3.2, iterations=200)
    except: pos = nx.circular_layout(G)
    vis = [(u,v,d) for u,v,d in G.edges(data=True) if d.get("pct",0)>=min_pct and u!=v]
    self_lp = [(u,u,G[u][u]) for u in G.nodes if G.has_edge(u,u) and G[u][u].get("pct",0)>=min_pct]
    if not vis: vis = [(u,v,d) for u,v,d in G.edges(data=True) if u!=v]
    max_pct  = max((d.get("pct",1) for _,_,d in vis), default=1)
    max_freq = max((G.nodes[n].get("freq",1) for n in G.nodes), default=1)
    for node, _, edge_d in self_lp:
        xy = pos[node]; clr = G.nodes[node].get("color","#888")
        off = 0.22
        ax.annotate("", xy=(xy[0]+off,xy[1]+off*0.5), xytext=(xy[0]+off*0.5,xy[1]+off),
                    arrowprops=dict(arrowstyle="-|>",connectionstyle="arc3,rad=0.80",
                                    color=clr,lw=1.8,alpha=0.68), zorder=4)
    for u,v,d in vis:
        pct = d.get("pct",1); lw = 0.8+(pct/max_pct)*4.5; alpha=0.25+(pct/max_pct)*0.65
        clr = G.nodes[u].get("color","#888")
        ax.annotate("", xy=pos[v], xytext=pos[u],
                    arrowprops=dict(arrowstyle="-|>",color=clr,lw=lw,alpha=alpha,
                                    connectionstyle="arc3,rad=0.18"), zorder=3)
        if show_pct:
            mx=(pos[u][0]+pos[v][0])/2; my=(pos[u][1]+pos[v][1])/2
            dx=pos[v][0]-pos[u][0]; dy=pos[v][1]-pos[u][1]; dist=max(_math.hypot(dx,dy),0.001)
            ox,oy=-dy/dist*0.09,dx/dist*0.09
            ax.text(mx+ox,my+oy,f"{pct:.0f}%",fontsize=6.5,ha="center",va="center",
                    color=clr,fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.10",fc="white",ec="none",alpha=0.82),zorder=5)
    for node in G.nodes:
        xy = pos[node]; freq = max(G.nodes[node].get("freq",1),1)
        r = 0.065+(freq/max_freq)*0.10; color = G.nodes[node].get("color","#888")
        ax.add_patch(plt.Circle(xy,r+0.022,color=color,alpha=0.16,zorder=6))
        ax.add_patch(plt.Circle(xy,r,color=color,alpha=0.92,zorder=7))
        ax.text(xy[0],xy[1],node,ha="center",va="center",fontsize=8,fontweight="bold",color="white",zorder=8)
    xs_p=[p[0] for p in pos.values()]; ys_p=[p[1] for p in pos.values()]; pad=0.38
    ax.set_xlim(min(xs_p)-pad,max(xs_p)+pad); ax.set_ylim(min(ys_p)-pad,max(ys_p)+pad)
    ax.set_aspect("equal")

fig1 = plt.figure(figsize=(20, 12))
gs1  = GridSpec(2, 4, figure=fig1, hspace=0.44, wspace=0.34)
fig1.patch.set_facecolor(FIG_BG)
ax_gen=fig1.add_subplot(gs1[0,0]); ax_pb=fig1.add_subplot(gs1[0,1])
ax_postb=fig1.add_subplot(gs1[0,2]); ax_hm=fig1.add_subplot(gs1[0,3])
ax_prs=fig1.add_subplot(gs1[1,0]); ax_posts=fig1.add_subplot(gs1[1,1])
ax_dhm=fig1.add_subplot(gs1[1,2]); ax_bg=fig1.add_subplot(gs1[1,3])

draw_seq_net(ax_gen,   G_SEQ_ALL,       f"Genel Sekansiyel Ağ\n({len(_pairs_all):,} geçiş)")
draw_seq_net(ax_pb,    G_SEQ_PRE_BUY,   f"Fon Alımı Öncesi\n(−{X_DAYS} gün)")
draw_seq_net(ax_postb, G_SEQ_POST_BUY,  f"Fon Alımı Sonrası\n(+{X_DAYS} gün)")
draw_seq_net(ax_prs,   G_SEQ_PRE_SELL,  f"Fon Satımı Öncesi\n(−{X_DAYS} gün)")
draw_seq_net(ax_posts, G_SEQ_POST_SELL, f"Fon Satımı Sonrası\n(+{X_DAYS} gün)")

sns.heatmap(TRANS_MAT_ALL*100, annot=True, fmt=".0f", cmap="Blues",
            linewidths=0.5, linecolor=GRID_CLR, ax=ax_hm,
            annot_kws={"size":9,"weight":"bold"}, cbar_kws={"shrink":0.82,"label":"Geçiş %"},
            xticklabels=URUNLER, yticklabels=URUNLER)
ax_hm.set_title("Geçiş Olasılık Matrisi\n(Tüm Müşteriler · Satır %)", fontsize=10, color=TITLE_CLR)

_delta = (TRANS_MAT_POST_BUY - TRANS_MAT_PRE_BUY)*100
_vmax  = max(abs(_delta.values).max(), 0.5)
sns.heatmap(_delta, annot=True, fmt="+.0f", cmap="RdYlGn", vmin=-_vmax, vmax=_vmax, center=0,
            linewidths=0.5, linecolor=GRID_CLR, ax=ax_dhm,
            annot_kws={"size":9}, cbar_kws={"shrink":0.82,"label":"Δ%"},
            xticklabels=URUNLER, yticklabels=URUNLER)
ax_dhm.set_title("Post-Buy − Pre-Buy Δ\n(Yeşil=artan geçiş)", fontsize=10, color=TITLE_CLR)

if not TOP_BIGRAMS.empty:
    _bg = TOP_BIGRAMS.head(12)
    _bg_clrs = [URUN_RENKLER.get(r.split(" → ")[0],"#888") for r in _bg["label"]]
    ax_bg.barh(range(len(_bg)),_bg["count"].values,color=_bg_clrs,alpha=0.85,edgecolor="white")
    ax_bg.set_yticks(range(len(_bg))); ax_bg.set_yticklabels(_bg["label"].values,fontsize=8.5)
    ax_bg.invert_yaxis()
ax_bg.set_title("En Sık 12 İkili Geçiş",fontsize=10,color=TITLE_CLR)
ax_bg.set_xlabel("Frekans",fontsize=8.5); style_axes(ax_bg)

style_fig(fig1, "E — Sekansiyel İşlem Ağı · Geçiş Olasılıkları  ·  2 Dönem",
          f"{len(_pairs_all):,} geçiş · eşik ≥{NET_SEQ_MIN_EDGE} · görsel ≥%{NET_SEQ_MIN_PCT:.0f}")
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

# Segment ağları
_ncols_seg = min(len(SEGMENT_SIRASI), 4)
_nrows_seg = _math.ceil(len(SEGMENT_SIRASI)/_ncols_seg)
fig2 = plt.figure(figsize=(18, 5.5*_nrows_seg + 1.5)); fig2.patch.set_facecolor(FIG_BG)
gs2 = GridSpec(_nrows_seg, _ncols_seg, figure=fig2, hspace=0.52, wspace=0.36)
for _idx, _seg in enumerate(SEGMENT_SIRASI):
    _ax = fig2.add_subplot(gs2[_idx//_ncols_seg, _idx%_ncols_seg])
    draw_seq_net(_ax, G_SEQ_SEG.get(_seg), f"{SEG_SHORT[_idx]}", min_pct=1.5)
    for _sp in _ax.spines.values():
        _sp.set_edgecolor(SEG_RENK[_seg]); _sp.set_linewidth(2.4); _sp.set_visible(True)
    _ax.set_facecolor(AXES_BG)
style_fig(fig2, "E2 — Segment Bazlı Sekansiyel Ağlar  ·  2 Dönem")
plt.tight_layout(rect=[0,0,1,0.965])
plt.show()


# ══════════════════════════════════════════════════════════════════
# § F — STRATEJİK İÇGÖRÜLER
# ══════════════════════════════════════════════════════════════════
en_aktif_seg   = DAVRANIS_SKOR["Aktivite_Skoru"].idxmax()
en_sadik_seg   = SADAKAT["Cok_Donem_Pct"].idxmax()
en_yuksek_aum  = AUM.idxmax()
alim_artis_seg = (GIRIS_POST_BUY - GIRIS_PRE_BUY).idxmax()
pre_buy_vadesiz_ratio = URUN_PRE_BUY["Vadesiz"].mean() if "Vadesiz" in URUN_PRE_BUY.columns else 0
post_sell_doviz_ratio = URUN_POST_SELL["Döviz"].mean()  if "Döviz"   in URUN_POST_SELL.columns else 0
gecis_diagonal_alim   = np.diag(GECIS_ALIM.values).mean()

print("="*72)
print("  BANKA FON MÜŞTERİ DAVRANIŞ ANALİZİ  —  2025.09 & 2026.03")
print("="*72)
print(f"""
  TEMEL KPI'LAR
  ─────────────────────────────────────────────────────
  Toplam Fon Alım   : {KPI['toplam_alim_m']:>7.1f} M TL
  Toplam Fon Satım  : {KPI['toplam_satim_m']:>7.1f} M TL
  Net AUM           : {KPI['net_aum_m']:>7.1f} M TL
  Toplam Olay       : {KPI['toplam_event']:>7,}
  Penetrasyon Ort.  : %{KPI['penetrasyon_ort']:.1f}
  Sadakat Ort.      : %{KPI['sadakat_ort']:.1f}
  ─────────────────────────────────────────────────────
  En Aktif Segment  : {en_aktif_seg}  (Skor: {DAVRANIS_SKOR.loc[en_aktif_seg,'Aktivite_Skoru']:.0f}/100)
  En Sadık Segment  : {en_sadik_seg}  (%{SADAKAT.loc[en_sadik_seg,'Cok_Donem_Pct']:.0f} her iki döneme katıldı)
  En Yüksek AUM     : {en_yuksek_aum}  ({AUM.get(en_yuksek_aum,0)/1e6:.1f}M TL)
  Pre-Buy Vadesiz % : {pre_buy_vadesiz_ratio:.1f}%  (nakit birikim sinyali)
  Post-Sell Döviz % : {post_sell_doviz_ratio:.1f}%  (re-investment kur riski)
  Geçiş kalıcılığı  : {gecis_diagonal_alim:.0f}%  (aynı ürün grubu devamı)
""")
for seg in SEGMENT_SIRASI:
    pen = PENETRASYON.loc[seg,"Alim_Pct"]; sad = SADAKAT.loc[seg,"Cok_Donem_Pct"]
    delta = GIRIS_POST_BUY.get(seg,0) - GIRIS_PRE_BUY.get(seg,0)
    print(f"  [{seg}]  Pen %{pen:.0f} · Sadakat %{sad:.0f} · Δ{delta:+.1f}")
print(f"\n  ✅ Analiz: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")


# ══════════════════════════════════════════════════════════════════
# § G — t-1 BAKİYE HEATMAPLER
# ══════════════════════════════════════════════════════════════════
if not ALIM_BAK_HM.empty:
    _yt = [SEG_SHORT[SEGMENT_SIRASI.index(s)] for s in SEGMENT_SIRASI]
    _hm_base = dict(linewidths=0.4, linecolor="#E2E8F0", yticklabels=_yt)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
    style_fig(fig, "G — t-1 Dönemi Bakiyesine Göre Alım / Satım Oranları  ·  2025.09→2026.03",
              "Alım/Satım tutarının t-1 bakiyesine oranı (Medyan %)  ·  İlk dönem = referans")

    ax = axes[0,0]
    _vmax_a = max(float(ALIM_BAK_HM.values.max()), 0.1)
    sns.heatmap(ALIM_BAK_HM, ax=ax, cmap="Blues", vmin=0, vmax=_vmax_a,
                annot=True, fmt=".1f", cbar_kws={"label":"Alım/Bakiye(%)","shrink":0.78}, **_hm_base)
    ax.set_title("Alım / t-1 Bakiye (%) · Medyan"); ax.set_xlabel("Dönem")
    ax.tick_params(axis="x", rotation=15, labelsize=8.5); ax.tick_params(axis="y", labelsize=7.5)

    ax = axes[0,1]
    _vmax_s = max(float(SATIM_BAK_HM.values.max()), 0.1)
    sns.heatmap(SATIM_BAK_HM, ax=ax, cmap="Reds", vmin=0, vmax=_vmax_s,
                annot=True, fmt=".1f", cbar_kws={"label":"Satım/Bakiye(%)","shrink":0.78}, **_hm_base)
    ax.set_title("Satım / t-1 Bakiye (%) · Medyan"); ax.set_xlabel("Dönem")
    ax.tick_params(axis="x", rotation=15, labelsize=8.5); ax.tick_params(axis="y", labelsize=7.5)

    ax = axes[1,0]
    _amax = max(float(abs(ALIM_BAK_DELTA.values).max()), 0.1)
    sns.heatmap(ALIM_BAK_DELTA, ax=ax, cmap="RdYlGn", center=0, vmin=-_amax, vmax=_amax,
                annot=True, fmt=".1f", cbar_kws={"label":"Δ pp","shrink":0.78}, **_hm_base)
    ax.set_title("Alım/Bakiye Değişimi (pp · ΔPoP)"); ax.set_xlabel("Dönem")
    ax.tick_params(axis="x", rotation=15, labelsize=8.5); ax.tick_params(axis="y", labelsize=7.5)

    ax = axes[1,1]
    _smax = max(float(abs(SATIM_BAK_DELTA.values).max()), 0.1)
    sns.heatmap(SATIM_BAK_DELTA, ax=ax, cmap="RdYlGn", center=0, vmin=-_smax, vmax=_smax,
                annot=True, fmt=".1f", cbar_kws={"label":"Δ pp","shrink":0.78}, **_hm_base)
    ax.set_title("Satım/Bakiye Değişimi (pp · ΔPoP)"); ax.set_xlabel("Dönem")
    ax.tick_params(axis="x", rotation=15, labelsize=8.5); ax.tick_params(axis="y", labelsize=7.5)

    plt.show()
    print("✅ t-1 Bakiye heatmapler gösterildi")
else:
    print("⚠️  t-1 hesaplama için yeterli dönem yok (ilk dönemde t-1 bakiye yoktur).")


# ══════════════════════════════════════════════════════════════════
# § H — FON-GEÇMEYEN SEKANS HEATMAPLER  (2 dönem)
# ══════════════════════════════════════════════════════════════════
_yt  = [SEG_SHORT[SEGMENT_SIRASI.index(s)] for s in SEGMENT_SIRASI]
_xt  = [FG_TURLER_KISALT.get(t,t) for t in FG_TURLER]
_hm_kw = dict(linewidths=0.3, linecolor="#E2E8F0", annot=True, fmt=".0f", yticklabels=_yt)

fig1, axes1 = plt.subplots(2, 2, figsize=(22, 14), constrained_layout=True)
style_fig(fig1, "H — Fon-Geçmeyen İşlem Türleri — Pre/Post Alım & Satım  (Segment · %)",
          "Mavi=Pre-Buy · Yeşil=Post-Buy · Turuncu=Pre-Sell · Mor=Post-Sell  ·  2 dönem konsolide")

_panels = [
    (axes1[0,0], DIST_PRE_BUY,   "#1D4ED8", "Pre-Buy  — Fon-Geçmeyen İşlem Türü (%)"),
    (axes1[0,1], DIST_POST_BUY,  "#059669", "Post-Buy — Fon-Geçmeyen İşlem Türü (%)"),
    (axes1[1,0], DIST_PRE_SELL,  "#D97706", "Pre-Sell  — Fon-Geçmeyen İşlem Türü (%)"),
    (axes1[1,1], DIST_POST_SELL, "#7C3AED", "Post-Sell — Fon-Geçmeyen İşlem Türü (%)"),
]
for ax, data, base_color, ttl in _panels:
    cmap_custom = sns.light_palette(base_color, as_cmap=True)
    sns.heatmap(data, ax=ax, cmap=cmap_custom, cbar_kws={"label":"% payı","shrink":0.78}, **_hm_kw)
    ax.set_title(ttl); ax.set_xticklabels(_xt, rotation=30, ha="right", fontsize=8)
    ax.tick_params(axis="y", labelsize=7.5)
plt.show()

# Dönem bazlı 2 sütun
_empty = pd.DataFrame(0.0, index=SEGMENT_SIRASI, columns=FG_TURLER)
fig2, axes2 = plt.subplots(2, 2, figsize=(22, 12), constrained_layout=True)
style_fig(fig2, "H2 — Fon-Geçmeyen Sekans — Dönem Bazlı Post−Pre Farkı (pp)  ·  2025.09 & 2026.03",
          "Yeşil=Post sonrası artış · Kırmızı=azalış  |  Üst: Alım · Alt: Satım")

for col_i, d in enumerate(DONEM_SIRASI):
    diff_buy = (DIST_POST_BUY_D.get(d,_empty) - DIST_PRE_BUY_D.get(d,_empty)).fillna(0)
    _amax_b  = max(float(abs(diff_buy.values).max()), 0.1)
    ax = axes2[0, col_i]
    sns.heatmap(diff_buy, ax=ax, cmap="RdYlGn", center=0, vmin=-_amax_b, vmax=_amax_b,
                annot=True, fmt=".0f", linewidths=0.3, linecolor="#E2E8F0",
                yticklabels=(_yt if col_i==0 else False))
    ax.set_title(f"{d}  ·  Post-Buy − Pre-Buy (pp)", fontsize=9)
    ax.set_xticklabels(_xt, rotation=35, ha="right", fontsize=7)
    if col_i==0: ax.set_ylabel("Alım Sekansı", fontsize=9, fontweight="bold")

    diff_sell = (DIST_POST_SELL_D.get(d,_empty) - DIST_PRE_SELL_D.get(d,_empty)).fillna(0)
    _amax_s   = max(float(abs(diff_sell.values).max()), 0.1)
    ax = axes2[1, col_i]
    sns.heatmap(diff_sell, ax=ax, cmap="PuOr", center=0, vmin=-_amax_s, vmax=_amax_s,
                annot=True, fmt=".0f", linewidths=0.3, linecolor="#E2E8F0",
                yticklabels=(_yt if col_i==0 else False))
    ax.set_title(f"{d}  ·  Post-Sell − Pre-Sell (pp)", fontsize=9)
    ax.set_xticklabels(_xt, rotation=35, ha="right", fontsize=7)
    if col_i==0: ax.set_ylabel("Satım Sekansı", fontsize=9, fontweight="bold")

plt.show()
print("✅ Fon-geçmeyen sekans analizi tamamlandı")


# ══════════════════════════════════════════════════════════════════
# § I — PROFESYONEL KOMPOZİT NETWORK  (Yapılandırılmış Hiyerarşik)
# ══════════════════════════════════════════════════════════════════

def _structured_layout(G):
    """
    Yapılandırılmış ızgara düzeni:
    - X ekseni: Ürün grubu (Vadesiz=0, Vadeli=1, Yatırım=2, Döviz=3, Kredi=4) * X_SPACING
    - Y ekseni: +3.5 = Giriş, -3.5 = Çıkış
    - Aynı ürün+yön grubundaki node'lar yatayda yayılır
    """
    from collections import defaultdict
    URUN_ORD  = ["Vadesiz","Vadeli","Yatırım","Döviz","Kredi"]
    X_SPACING = 5.5
    PROD_X    = {p: i * X_SPACING for i, p in enumerate(URUN_ORD)}

    groups = defaultdict(list)
    for node in G.nodes():
        parts = str(node).split("|")
        prod  = parts[0] if parts else "Other"
        yon   = parts[2] if len(parts) > 2 else "Other"
        groups[(prod, yon)].append(node)

    pos = {}
    for (prod, yon), nodes in groups.items():
        x_base = PROD_X.get(prod, len(URUN_ORD) * X_SPACING)
        y_base = 3.5 if yon == "Giriş" else -3.5
        y_dir  = 1   if yon == "Giriş" else -1
        n      = len(nodes)
        for j, node in enumerate(sorted(nodes)):
            x_off = (j - (n - 1) / 2.0) * 1.3
            y_off = j * 1.8 * y_dir if n > 1 else 0
            pos[node] = (x_base + x_off, y_base + y_off)
    for node in G.nodes():
        if node not in pos:
            pos[node] = (len(URUN_ORD) * X_SPACING + 2, 0.0)
    return pos


def _bezier_pts(x0, y0, x1, y1, n=50, strength=0.38):
    """Cubic Bezier eğri noktaları üretir."""
    dx, dy = x1 - x0, y1 - y0
    dist   = max((dx**2 + dy**2)**0.5, 1e-9)
    nx_, ny_ = -dy / dist, dx / dist
    cx0 = x0 + dx*0.33 + nx_ * dist * strength
    cy0 = y0 + dy*0.33 + ny_ * dist * strength
    cx1 = x0 + dx*0.67 + nx_ * dist * strength
    cy1 = y0 + dy*0.67 + ny_ * dist * strength
    t   = np.linspace(0, 1, n)
    bx  = (1-t)**3*x0 + 3*t*(1-t)**2*cx0 + 3*t**2*(1-t)*cx1 + t**3*x1
    by  = (1-t)**3*y0 + 3*t*(1-t)**2*cy0 + 3*t**2*(1-t)*cy1 + t**3*y1
    return bx.tolist(), by.tolist()


def build_professional_network(G, title="Profesyonel İşlem Ağı",
                                max_nodes=32, top_arrows=50):
    """
    Yapılandırılmış hiyerarşik layout:
    • Ürünler X ekseninde ayrışır (çakışma yok)
    • Giriş/Çıkış Y ekseninde ayrışır
    • Kenarlar Bezier eğrileri (iç içe değil)
    • Beyaz daire + renkli kenarlık
    • Sade ızgara çizgileri, ürün başlıkları, yön etiketleri
    """
    URUN_ORD  = ["Vadesiz","Vadeli","Yatırım","Döviz","Kredi"]
    X_SPACING = 5.5
    PROD_X    = {p: i * X_SPACING for i, p in enumerate(URUN_ORD)}

    if G.number_of_nodes() == 0:
        return go.Figure().update_layout(title=title, paper_bgcolor="white")

    # Subgraph with top-N nodes
    top_nodes = sorted(G.nodes(), key=lambda n: G.nodes[n].get("freq",0), reverse=True)[:max_nodes]
    Gs = G.subgraph(top_nodes).copy()

    # PageRank
    try:
        pr = nx.pagerank(Gs, weight="weight", max_iter=300) if Gs.number_of_edges()>0 else {n:1/max(Gs.number_of_nodes(),1) for n in Gs.nodes()}
    except:
        pr = {n: Gs.degree(n) for n in Gs.nodes()}
    nx.set_node_attributes(Gs, pr, "_pr")

    pos = _structured_layout(Gs)
    traces = []

    # ── Kenar izleri (Bezier eğrisi) ──────────────────────────────
    all_edges   = sorted(Gs.edges(data=True), key=lambda e: e[2].get("weight",0), reverse=True)
    max_w       = max((e[2].get("weight",0) for e in all_edges), default=1)

    for src, dst, edata in all_edges:
        if src not in pos or dst not in pos: continue
        x0, y0 = pos[src]; x1, y1 = pos[dst]
        w     = edata.get("weight", 1)
        pct   = edata.get("pct", 0.0)
        alpha = round(max(0.15, min(0.72, 0.15 + (w/max_w)*0.57)), 2)
        width = round(max(0.8, (w/max_w)*6.5), 2)
        src_prod = str(src).split("|")[0]
        clr  = URUN_RENKLER.get(src_prod, "#94A3B8")

        if src == dst:  # self-loop: küçük çember
            theta = np.linspace(0, 2*np.pi, 44)
            r = 0.75
            lx = (x0 + r * np.cos(theta)).tolist()
            ly = (y0 + r * np.sin(theta)).tolist()
        else:
            lx, ly = _bezier_pts(x0, y0, x1, y1, n=50, strength=0.35)

        traces.append(go.Scatter(
            x=lx, y=ly, mode="lines",
            line=dict(width=width, color=f"rgba{tuple(int(clr.lstrip('#')[i:i+2],16) for i in (0,2,4))+( alpha,)}"),
            hovertemplate=(f"<b>{str(src).replace('|',' | ')}</b> → "
                           f"<b>{str(dst).replace('|',' | ')}</b><br>"
                           f"Geçiş: {int(w):,}  Pay: {pct:.1f}%<extra></extra>"),
            showlegend=False,
        ))

    # ── Ok başlıkları ─────────────────────────────────────────────
    arrows = []
    for src, dst, edata in all_edges[:top_arrows]:
        if src == dst or src not in pos or dst not in pos: continue
        x0,y0 = pos[src]; x1,y1 = pos[dst]
        lx, ly = _bezier_pts(x0, y0, x1, y1, n=50, strength=0.35)
        if len(lx) < 4: continue
        axe, aye = lx[-3], ly[-3]
        bxe, bye = lx[-1], ly[-1]
        adx, ady = bxe-axe, bye-aye
        adist = max((adx**2+ady**2)**0.5, 1e-9)
        node_r = 0.55
        bxe_s = bxe - adx/adist*node_r
        bye_s = bye - ady/adist*node_r
        src_prod = str(src).split("|")[0]
        clr = URUN_RENKLER.get(src_prod, "#94A3B8")
        w   = edata.get("weight",1)
        arrows.append(dict(
            x=bxe_s, y=bye_s, ax=axe, ay=aye,
            xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=4, arrowsize=0.9,
            arrowwidth=max(0.5, (w/max_w)*3.0),
            arrowcolor=clr + "CC",
        ))

    # ── Node'lar (ürün grubuna göre grupla) ───────────────────────
    from collections import defaultdict as _dd
    urun_groups = _dd(list)
    for n in Gs.nodes():
        urun_groups[str(n).split("|")[0]].append(n)

    node_freqs = {n: Gs.nodes[n].get("freq",1) for n in Gs.nodes()}
    max_freq   = max(node_freqs.values(), default=1)

    for urun in URUN_ORD + ["Other"]:
        nlist = urun_groups.get(urun, [])
        if not nlist: continue
        border_clr = URUN_RENKLER.get(urun, "#94A3B8")
        sizes = [round(42 + 46*(node_freqs[n]/max_freq)**0.6) for n in nlist]
        hover = []
        for n in nlist:
            parts = n.split("|")
            hover.append(f"<b>{n.replace('|',' | ')}</b><br>"
                         f"Ürün: {parts[0]}<br>"
                         f"Tür: {parts[1] if len(parts)>1 else ''}<br>"
                         f"Yön: {parts[2] if len(parts)>2 else ''}<br>"
                         f"Frekans: {node_freqs[n]:,}<br>"
                         f"PageRank: {pr.get(n,0):.4f}")
        labels = [_short_comp_label(n) for n in nlist]
        traces.append(go.Scatter(
            x=[pos[n][0] for n in nlist],
            y=[pos[n][1] for n in nlist],
            mode="markers+text",
            name=urun,
            text=labels,
            textposition="middle center",
            textfont=dict(size=8, color=border_clr, family="DejaVu Sans"),
            hovertext=hover,
            hovertemplate="%{hovertext}<extra></extra>",
            marker=dict(
                size=sizes, sizemode="diameter",
                color="white",
                line=dict(color=border_clr, width=3.2),
                symbol="circle",
            ),
        ))

    # ── Arka plan şekilleri ────────────────────────────────────────
    shapes = []
    for i, prod in enumerate(URUN_ORD):
        x_c = PROD_X[prod]
        # Ürün grup bandı (çok hafif arkaplan)
        shapes.append(dict(
            type="rect",
            x0=x_c - 2.0, x1=x_c + 2.0, y0=-8.5, y1=8.5,
            fillcolor=URUN_RENKLER[prod] + "0A",
            line=dict(width=0),
        ))
        if i > 0:  # Dikey ayırıcı çizgiler
            shapes.append(dict(
                type="line",
                x0=x_c - 2.5, x1=x_c - 2.5, y0=-8.5, y1=8.5,
                line=dict(color="#E2E8F0", width=1.2, dash="dot"),
            ))
    # Giriş/Çıkış yatay ayırıcı
    shapes.append(dict(
        type="line",
        x0=-3, x1=PROD_X.get("Kredi", 20) + 3, y0=0, y1=0,
        line=dict(color="#CBD5E1", width=1.5, dash="dash"),
    ))

    # ── Annotasyonlar ─────────────────────────────────────────────
    annotations = list(arrows)
    annotations += [
        dict(x=-2.5, y=7.8, text="<b>↑ GİRİŞ İŞLEMLERİ</b>", showarrow=False,
             font=dict(size=12, color="#059669"), xanchor="left"),
        dict(x=-2.5, y=-7.8, text="<b>↓ ÇIKIŞ İŞLEMLERİ</b>", showarrow=False,
             font=dict(size=12, color="#DC2626"), xanchor="left"),
    ]
    for prod in URUN_ORD:
        annotations.append(dict(
            x=PROD_X[prod], y=9.5, text=f"<b>{prod}</b>",
            showarrow=False,
            font=dict(size=13, color=URUN_RENKLER[prod]),
            xanchor="center",
        ))

    xs_all  = [p[0] for p in pos.values()]
    ys_all  = [p[1] for p in pos.values()]
    x_pad = 3; y_pad = 1.5

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center",
                   font=dict(size=15, family="DejaVu Sans", color="#0F172A")),
        paper_bgcolor="white", plot_bgcolor="white",
        showlegend=True,
        legend=dict(title=dict(text="Ürün Grubu", font=dict(size=11)),
                    bgcolor="rgba(255,255,255,0.95)",
                    bordercolor="#E2E8F0", borderwidth=1,
                    x=1.01, y=0.98, font=dict(size=10)),
        xaxis=dict(visible=False, range=[min(xs_all)-x_pad, max(xs_all)+x_pad]),
        yaxis=dict(visible=False, scaleanchor="x", scaleratio=1,
                   range=[min(ys_all)-y_pad, max(ys_all)+y_pad]),
        height=780, width=1200,
        shapes=shapes, annotations=annotations,
        margin=dict(l=30, r=220, t=90, b=40),
    )
    return fig


# ── Tüm segment kompozit network ──────────────────────────────────
print("\n▶  Profesyonel kompozit network oluşturuluyor...")
fig_net = build_professional_network(
    G_COMPOSITE,
    title=(
        "Tüm Segmentler — Kompozit İşlem Akış Networkü  ·  2025.09 & 2026.03<br>"
        "<sup>Her node = Ürün | İşlem Türü | Yön  ·  Boyut ∝ Frekans  ·  "
        "Kenar kalınlığı ∝ Geçiş sıklığı  ·  Bezier eğrili kenarlar</sup>"
    ),
)
fig_net.show()

HTML_DIR = "."
HTML_ALL = os.path.join(HTML_DIR, "islem_network_tum.html")
fig_net.write_html(HTML_ALL, include_plotlyjs="cdn", full_html=True,
                   config={"scrollZoom": True, "displayModeBar": True})
_abs_all = os.path.abspath(HTML_ALL)
print(f"✅ HTML kaydedildi: {_abs_all}")

display(HTML(f"""
<div style="padding:12px 16px;background:#F0F9FF;border:1px solid #BAE6FD;
            border-radius:10px;display:inline-block;margin:8px 0">
  <a href="{HTML_ALL}" target="_blank"
     style="font-size:15px;font-weight:bold;color:#0369A1;text-decoration:none">
    🌐 Profesyonel Network Grafiğini Aç (Tüm Segmentler)
  </a>
  <span style="margin-left:12px;color:#64748B;font-size:11px">{_abs_all}</span>
</div>
"""))

# ── Segment bazlı HTML'ler ──────────────────────────────────────────
print("\n▶  Segment bazlı profesyonel networkler...")
_seg_links = []
for seg in SEGMENT_SIRASI:
    Gs = G_COMPOSITE_SEG.get(seg)
    if Gs is None or Gs.number_of_nodes() == 0: continue
    figs = build_professional_network(
        Gs,
        title=f"{seg.replace('_',' ')} — İşlem Ağı  ·  2025.09 & 2026.03<br>"
              f"<sup>Yapılandırılmış hiyerarşik layout  ·  Bezier kenarlar</sup>",
    )
    fname = os.path.join(HTML_DIR, f"network_{seg}.html")
    figs.write_html(fname, include_plotlyjs="cdn", full_html=True,
                    config={"scrollZoom": True, "displayModeBar": True})
    _seg_links.append((seg, fname))
    print(f"  ✓ {seg}: {Gs.number_of_nodes()} node · {Gs.number_of_edges()} kenar")

_link_html = (
    '<div style="padding:14px;background:#F8FAFC;border:1px solid #E2E8F0;'
    'border-radius:10px;max-width:950px">'
    '<b style="color:#0F172A;font-size:13px">📂 Segment Bazlı Profesyonel Network HTML:</b>'
    '<br><br>')
for seg, fname in _seg_links:
    clr = SEG_RENK.get(seg, "#64748B")
    _link_html += (f'<a href="{fname}" target="_blank" '
                   f'style="display:inline-block;margin:4px 5px;padding:6px 14px;'
                   f'background:white;border:2px solid {clr};border-radius:7px;'
                   f'color:{clr};font-weight:bold;font-size:11px;text-decoration:none">'
                   f'{seg.replace("_"," ")}</a>')
_link_html += "</div>"
display(HTML(_link_html))
print(f"\n✅ {len(_seg_links)} segment network HTML hazır")
print(f"\n{'═'*60}")
print("  ✅  PART 2 TAMAMLANDI")
print(f"{'═'*60}")
print("  Chartlar    : A (Genel) · B (Alım) · C (Satım) · D (Dönemsel)")
print("  Ağlar       : E (Sekans) · I (Profesyonel Kompozit)")
print("  Heatmapler  : G (t-1 Bakiye) · H (Fon-Geçmeyen Sekans)")
print(f"  HTML        : {len(_seg_links)+1} dosya oluşturuldu")
